<!-- dd:dd-lesson-np-2 -->

# Indexing and selection

*Numpy · `np-2`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "np-2"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrtPdtu47iSvyI0sGj7HMvDiy5WsPO6+wPzljYa7sSZyY7bDmx3TiaN+fdl8a6LLYmiROfgNNJORMliVbFIVhXr8vMTodmn"
    "u+jnp6c9+/XpdNi9bj8tok8Pm9P2xFruf346bc8/Xr4+HB638MTz95fD8RydD8eHP6LNKTp/2b9Fv0bn5Xm7Px2Os/t7vESL"
    "KIUPskTrRXSfw99omS4iyhrW8y/7hx/nw9MT+xpZpqK33Q5ePtttvn973ETHu2j2fHren86b/cN2dlyw9//G3z9fRLPT+Tg7"
    "Lzenr7LL43z5eP7rZTuHx0rN58Pu+XSezdm/GUdt9raIROfzOXS8fXvZPpy3j1/ZH0cOwW/HH1vWx2eO4PLb4bD7zFC4/5/N"
    "7sTaxV1+AZhZl+Iew27+6e9F1J9qFIhElym8ljTSijV8OFoBNvJSEM2ZPjEBAsVIUCimQCIk+CvmHBcnS5KWKQY3PjLJ/FIw"
    "ASpxUmXwkVT5K/lgxGqYks60QVViMNb5ICuTWXbWDPVPKRm6nL+XSFNb4iRbzrtR53l/nvNPhrRG9f06irTrIJYhLc2SEcFD"
    "buBxqkX/iPIRQcsHUG5sknHuxJ64c3Pc7H/fzjCe95+kF2biInpvmJUtiM3uYU1lkg7jWtiC2H+2L8UZ+58votUiKthtvknZ"
    "z7HH2FPsIfuZudvYUf4K9vZi3Z8WFxelbtiXO3eB3lAiAPTl4XLEIBYoYMJHkwREA0CIJQwumGQhYIdOxdKwSv1uXDDj4Acm"
    "H+Z/EjkPVauciesAeNsQWWC4DNu9Qof/DoCLK9ww8zBXd4AQIQahBoEr/fNApLf7FrOIUOJXnad8eSvE8oITEJIfQRmdXDh+"
    "7CcXX5BbtXzopijwrSLlczcXu7YgR3rr5LhECCXNu5EjV/iTD4G/K5oJF1GYpIOxQhh/DP6vfvalgJKzGepUYp58EFa/aCXo"
    "ecuo1jgZLKJsSnyFhc1PWEqlzRTMMh42EkZvpqJlScOmsmnZVNhg0857YRkhhYHBKjAujnggPSYo/HD0wmE205OWMPD5NPv1"
    "vHzaHTZnSuZRzDatefRLxCA5bk9/bF7guQ85RliPEZGGzRvgNSGDIc82OJiS8CNVGTAiwJ8JVxwSo9UEU2UURBYYTqJ0DoYA"
    "ODGC37euypiZRtmU+q8ojf4ZITOtMOKWzBAqAS5plqjONkgykzRHWfapXl9zHWb15clpo6Hm87RIfE9T69CT2mcsf1VPGwrY"
    "WTgt0TLjh6P8ufMfwD2HHchZyOUAop9tcxH9xb6u+myzdMpzNmIfunEE+SLQjLobg6T6kCpfojr9kNyY5e48nGytfNWHUFXo"
    "B1GgjjyR0lUVb5ejTr94y2NY7Iz0fSGnQW26+JkdftGV0CrdAKERdANiMfm3OhsgzQsDaWFknEX0rQVvJE/elWzsIrsVnFOy"
    "JsS8KQ59cOKdxtQZn1TOdvYS0oRUZcWadLTQ4NGKV804pZNiEuNMYCBk7AT5tXPiJeEskEtXpZh/EulZ4nHQ3lrw1F1KZ43Y"
    "rAQSECfDJRM3OHemwKhFUXhmxbdumwOfaKu+7FhDhaTTQu4IKVXLnJyCWHAW9jp13jqtbRYQkquwNZv8TCbQIag4XhPHEsBw"
    "6wt43tvobMxuHD0djtEmet5Hx3Ubilz3BOPset1jfMSxD+g6o4KmFKpeoCFFPjQucP2gSkamFEAjzoAzPxrZ+/Z4OM1mcKw6"
    "N0YwYdhhz3C1dg1CM1xwpdDJpO/FdKI07r7atMQRdPEajh/mDJiYs+s4D3WCqiwd2NmgoexzIRFQxhmkzrKFXoKR1ykFrjqN"
    "U4rcgc/yXXZtJrGOdtsZuI69iiXilS8R7bgSvszDUbTTHKFNc0SuA3h95fjWGWDJEOynH8CH/VbM6aTrnHYGUexPqLe7paRp"
    "emltRWo5hQtuRxyDI5CwUQKhJKfT1VBGfy0f+mqxOxd/gRvEFZFZekRKwF8vAI470vq1QZA0h4Rx69nUmNBoWV6rSXF+TSTv"
    "Bgt1giVfDu0X9er3ePixf5xZp2yrhkO2f7JpH/0S4cycBqzm0T8iOBJ43D48f9/sTr/SwWALRRgXnh1+jNKZSxV4amUFa2WF"
    "OOtehVCgUzVZEqF5FtPhgpohWBXwL8OrlCKSU1clOJGUSrXhaip9ONGj08ti9la3X6b6A09uYUGVvp3cnyY1RCBLYS9Wfo+u"
    "uHUILWFLXa74Xym3uG+Ov2/PfcztzdEBDGf+plHiPjAypqVsWYY7TlHXc5EhkGNHyO2zHAUymQJgxxiWvELefAq2MMEj/r0q"
    "Cnm4nfNz7iD+EzDxXPVO4QGi1D5Qc0JgwDVPRwx4/AVY3gtuLAxBf2cvBiD7itsOAlEd9WUdyzksBZcVjMo+K1ypmxwT6cck"
    "vZnMCY/f6c53/cKXi2Vn5IywkQzzBjBh9XRK+FFT326BUmNJrO8jSayV8Civwt57N2GvB5zummkXpXS6eMolyjBXkVbFKssJ"
    "RlwoxYR7HeBVzu/RNCNZsaLcd0rcopgg+JclNKcIZ/xrNOf3WFP9a8JqOll3qXEkz7DfKFiE5sI+kdFl7o1DF9Hr9cECo6TL"
    "TCL6EFgI7amQKjkCg72YesDP+h0ut7/2Edl9QM06d1yAc0NnxjJTwgxdayk+W/mX40FwMXJ8kJMk4h55e1+os+IAYuQAJ14a"
    "Tuegrq7aBOTeoiz2srV7NT0KwkMCS7YRc4MWYyRjqvrlWk6q3LHy+bFinJOu3v5MW0z3f3xrDcdVMIFK1exP7BS8RkojTWWY"
    "h5KJ6uiLkzs6NfpUG4nTMt66oZBORNKLZ+1otry3zpByJQalMmYrxheYIpaGiklpUoFLMa6B3ZUEROSRymU6qeQC0pPPgXyZ"
    "OuNUaLkqr8/u+2tORSMh0wCQ4uIqmHIBzAdLxrvNt+3uVDZuc6FAhLIImW2z+7E9VYzI7DZIdWiEzUzAxKQk3m+bPYpDoaAR"
    "QeYd+aEBd7VFN2MttHGkz7fzZZ6Gxh9gEJBoeJyxxxdHmxM4+FgLENzRUxIEkv5HzdiCabUIjSoYeAth55W/OEzS1pf6FntK"
    "cWuQYkhqB5BuaQx5tX1hl32XIVJwui75MZWKD1KpTHLpUgjLSLwKgqrsugkqAa8rtlit4vxdgpMYUcNgaXUv3QclaK7YJSKB"
    "GAqCjd25m0wJVBCzLMz80v3bUVk082ppI2nlrIRJV4vo/+CAfOGSEqRvuCZ01hqmWUotV0s9p9YGjGVKH8rT+kAr5BBj38SQ"
    "2QaSnfAvgxhC4OvA2+xZkoiAz9pLa90O7WXu7oxrnKolYEIQ1kNFxogdbB8cMTY2bHhI4qlcrj0if6CFIV6M4czeDcMqUL1Q"
    "UycXJYWdGMRoQMSUf3UmcxQkcufV+7dQYJBf037VdvFl/7I9fm+w1lAfBptG4kCHbcSRM7+SSIDzAM+a5CHNXWocHWJiJYlo"
    "oIhW9UKRowyjWIPcowsSLmmQZlwDYmngGjjJub978ziGRU/lJi1tomayU3TrastlCeOtKddWq2xBhSlFroOp3MWEHJzLTBAt"
    "OLjK/khE7eUmjyd3JAgl/hMNjsjvu9LwuMvQkFwwC4IN5DXM3DUXrtqJJT+M9lIBwC0HfDUDTqq2e1uk4dtZIIbDMtQrFeFQ"
    "ku1yZUvAAr7Czkrh19OA1vIbfdn/CTkBv+yP+9/5k/+73W+PG/aG2Xz5fbP/sdl9PW23j7MExLj+65f1kIgVqtCRw8IeZC+V"
    "DxwP/5pDENFPcfkm4oreoMloVX/zRvYoNF8amkW02+5nP63Xtn/pbxjR8tn4n4uIEadlaRXSnUpGmHjXfWCUaPso5aMOErtk"
    "uM96EdSVnrBTsS7vIRQP/oAGCKoDvT2bOyuXwupTaJridppiF5p2kGG60MEG2EVIKzmKCVUMsE7bsab/me7X2DMVXCkdkUyK"
    "0Tz3nEZI2WjU9sGG5fvm5eWZD99PdMdTO9+BcEbuYCuhd1H2t3d23Swi2WsLv8r9zBiSHBPySJegMrbpXYQC4tbXY3vTcIqZ"
    "1AeQH2axIeQ5Le7gWIuPIlwmd1GCQqKsDtoWAIcj4rQBZY4w4fhSjm0SEEnjs45Xiddoex64Ls1fi+hVGi+5P8g0xuZWzz5R"
    "O8mE061rOaeUz4ZuEObj6jMtDe7WYMuRSOfOW5doKs5uinES6HWiYgOYxbK39fTpB9sVQZAEOQfcRG0kqczH45JV0nqIe03N"
    "jvdcz2X31DXiTgaLCJJwzyqsFv36a5VSv82XsIU3SHfduE4kshKf0NSTO7Z/bWekOq8wd0gJzQO5VTMR264x2HM8jDyuNF6c"
    "8y5Jfl4qSX5eykl+3q9lalJmIW1HcwuGEYYX8wuNDHmMZVZjYezH7qDnXKzJr1U78gOyOKBwhLIYGbhChsy5lrvC18osjcGs"
    "IrWPDkvDfiX0RLCU7LGeD1NFxq3GMYZ3yY4pw/JcUkemHC/4lUvt41tVqo1TeY4QBDlZzYW4ZqG/NG7CoTAMTo645HUs9IqF"
    "w2CSD9L/CFcgqzgVQoTm6mEInETnhWWk9VwuQAfTtg2dn8Uykb2IlYLIxdJhbRdHOvpHFhO4VtnQDwKpLGEo7KXUHYGxtyUy"
    "ZOcEMSYWO43+NYHkpA4Cid5BcbLyvoOKBHFYZnEYYVp3SIEgEtS5ZsSWZtoAoA/b+YRsEIbepG+lIrs4TlP+Ay+lcJxRqReG"
    "NW5Fhf+Yyao6Vi7sEbKMzND6IqnJ6alWdXMdCDcDyJCAS7nJmt+BqqjQ9YCMr6qGTJiSmSrlLFEuWWKC5WM7Cv85TorhRfRn"
    "e4KhXHrgFm6ngBewiYOgk0qvYnAEom74FAYdapwYUAhsRB7T1fDT2UT40P3pVJvVAyLUKkGbT1NXaaJcLKlrwoV7YZ9O6nG2"
    "JDUBrYXJzx8jiHebDDHkjNhqOSGYK9c0HVbuIn3yA8pIOf1jZhncV/pWIcPQrZIJmJjDqElHSJRoGDypXjfV8MESZbzhBP10"
    "OuqonfaRxuM/2uckrIaklfxJprNNdZGqCRG2wLh+0nnpVl4qu1T/FpH3L70wxoPIWExNsMIBXDtPNl9poXDRlEDTy4MYk2vD"
    "fnHsVvYRIB189LB3KuB8gSL7TrO89LEuZZSrtaLLrcg+EO3EFHsPAncvXPuBRiYfBpv4/WDN+vsMfOaeAZ/BVDik/HcLcldY"
    "rY3p2tivjRGv3kdd7ltjodXQlY8Znk5bon3frz77hdL0stq6KeDe+sCF59bXa8BffcSu+97d5chtpZmc5j0xIh+Fiyo80xPN"
    "5CNOlivT5NoEuTo1rGUIJ57DWJl8mjXbkMiwBOItZokUuxlWIJJmiJHIEVyc+oj7WPezNjrCSnyarGI6Ll2JyT4ZZoN1ld2u"
    "6Ia45RZuuYVbbmHL3HGLEje6XYm7aZj6weqwKzGRm4nbT9Ut6Pn0lYviz/vfv74cYOrMoy1b9KPPz5+9bFhPn5sZtZlHm9mz"
    "mTOtHSm5hWlbN4zvu1TnMzU9ZbmHUoPw6C83mEKqpkFWPJuW5R0xVohOu3S4ADv2rPTIQGXuKbNOmW8sp/EM+amKZwSjzP+Z"
    "7WuH8/MaHUhDA21oSBoaemQheW129wxGAwt7pwRyr1WXz+kxyV2hjYlaBnEIuOMyyzVfJrVLy8sArwZPx/0ieoQyghjMcqoc"
    "JPaXzF2/v1O25viKEBqbqiExvmIYv/RY2WzW9FiDYa3y2LXTEBGv1n39twhPZFLiPADdaTVKiLpiAQGisqrp5FgkffIc20BT"
    "YzedHGhUEk5L4m3NzIzsg5vCs4c36PmFdKTCsuSB9ixHfQ0Afh1BmgASloliSBKsbhiTIBg3oGmB6VxUXGdPhRiVlcYxiJ+S"
    "hEEllh2UzAypZGbhRsxAoeuk+w6v5DHvIyYV71AfzgTLuVUdS7tUmh61osqA0oIYoQ9UTkUMVSILBkOI0jILUVZFC60igMqa"
    "H5490FNdjk+HPYdcEXCfUAJeUmZm1d/qVmkNo5AOmipJ38DqffBZmMEK49G8Hlp4LNMFGWUCs0A8J20WepqlwzPhN0gRVKVY"
    "FkF6pf23NIAW6BqjSyHS6r3idbqDtfMBViqDJIjJuEd5MEzSLCz0Afb62x0lGUvut1J2DCMrsZxPdQdr5/zQqU4yLaLwisEA"
    "JpKOpZfr0L4V9cy+aMkVPl6dnOhcMnbhmCABJCYXMemt1Fg7B0ku7xy0nPo5C4BpNTRt3RCu1rnNuTZeoGF2D9RKpFVGOjTj"
    "QAhYSd+MBynxm7GYywPn5X6z7+VEPgP3lkW0f+F+Ll/nKnHQsXM+VJ70xynPa0y0J/V5+bx/gtWke2Xf4ZBzjxs30DmhPwKN"
    "0ZSsMICgqSxCm6qpuo1RIRl6cjrLHdTvBspdy+UExdPWDU8t06jbSirh7jUcniqeI/c65yK8K/dlN+le5rzW8QCai8pw8gKK"
    "JU5L/nJhOssKkXqXL5UFQnCZkuVLrFeqxSmaXHLxD3Ebfe/nNtrg3emaQLFKidJ1ALnG8mb1kBLSXzhpf0zsQVoPGR158hQM"
    "BytHIRkp9LccvSl5sBa6KRVnKdeVK+UOpswMXnvOEgcq9bYullYoEbCYljKYyqK1vIh9aOzWw1YWPR0Do4EGrCd8PDjbyUVF"
    "r5TSZvGROVCv+BI9cSYUUxttnxvCkDEsnVkQNEY19ZpM4DEe/K1TLF8yrC56dWCJtbLKv1eDJB1X5DILgho0ZGjB88puMSVi"
    "l/p3HzWVnMCsM1MyITLZEfRvM+181cC7Nu2yKdMwSAgya9plfdNzt1l9YwhSiX6JEmP1zUR8zPu9g8WYfzf6b24158RkxJuM"
    "WlL3z1Md2p6na2MTyFOl4LFPkhpfwVjeEyIPkvekPk6qb5OhAtDOX4ZlOxGrBVHJHohMqjJk7yPWTNPrbzFMHbyYj5+bj/Yv"
    "y+cTe/XsXXjUz3sog/cCNGKBaYNMRRJ+h7zp9QV1wjlo96lWGj8LzWG/1cUWJpwk17N+NITDDLvV0Jdzgj3jGoRLiY1gEidT"
    "Jgdqp2MZ1ipJbKhHIxc/YAhFEo1QLpfQxuoabjvZlc3IKu8Jjk3B+aGKORbEEJ/U2qqup2soSTok/3fIx4LrCSmut5bf0CuA"
    "Kg2I6yXUkJ9bQ0nzn1Q1lVQ1bmSkN5nOQa6On2uc2RhFUOUiHUaAvMs7ZgEXJpcv+6fn3W605J7w8lZPc5mo3PwXGsIi0j99"
    "myqvG+C7LiqOSxoVoUgks4VLV6xCJftXV441H02OXi7BaTTRMg2FqNyTxYeWM0m1QT/hlkc1K6X6IBbmwfBuiAjHpahfaq4S"
    "E35ifc8KOsy8JnOpZ8/lF4vo5XB6Pj8f9gJFkBTlEjMDgOaLkdaUxt67LzJqjVA+fbI4QrL2lVu4lTqwWrHf6GaIc70Strcc"
    "0tcIkwJhElEktbgZwmAwM2Ui+TRe8YTa0vGZAMGgnhosIEl5HR6yFIv9pusEuykWslavvLw8WRrU0JXpePjXaRE9HHj+TzYi"
    "K+8UMD10mzbVT5O2ofpp3GRv8DudGbY0ArCUhRwBR7BhPw3PONZoOOIBstHNJA7qiv+MvU9RQSeOkSxbu+7uxlTly+yGVgY7"
    "7jT7oJmNEl20EIQlU/yrVMZFY00anyi/498sWRCdHkJiaG2t8Kq1F/R5/3UElogyCqJUjMzYV7mFZAXh5lvZuoeRBfFArsxk"
    "0aWjKjwNkVMXoqVmWjDj6ywSgj5WFyDcEn6BxSNUXcAjibqAR7g8PCPisUxdwGO5uoDHVnP3WLGYVM4oOuKW27jFxBUCwap8"
    "mXCDA9uUJDa9qCtMqRskac/+6mYIEo7PdCwfJuOWRAoVz2apJrrYmqWr8OJCRBRMsq0vJZVP21mkvrf2k7c1TNU1GwBnJXV6"
    "0HtnVnn6sdvNRNbZ6vQa6LHx7uCxL+qoGdajqhK7YjpVl91O7UfBcePePEtL37x+ZWWZ96h2MwodXr7utk9nqYCPqwGa7rrl"
    "y/tQOvfl7ziog6WBkQbjmxgYP9puA3o3xXfWYA5GTxxn3cjoDcZGZJe5kcHiwyTKKvO/LMXcq2kSdp2wBohyslT5I9VD+VPb"
    "MBxNLelNYOpuxaShrZiKH8mU9swbsgNWGNWuTJENztP2UNa5vA/0g0taDWMoMwa1kv3MyHM1s5q2ollbT8pfXUkB0YmRHmpp"
    "7EZLYffQHmige1cJhniLzjvEW1xxS0OglLpCi8SBdSIinEIwre7enJjzJsmDuknyrmqSe+rf/w+k64ub"
)
print("Delta Drills checker ready — 66 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-numpy-boolean-masking -->

## Boolean masks — compare, count, filter

`numpy.boolean-masking`


### the comparison IS the mask


A comparison applied to an array is itself elementwise: `x > 2.5` produces a
**boolean array** the same shape as `x` — `True` where the condition holds.
That boolean array is called a **mask**.

Saying "the comparison IS the mask" kills the urge to loop: there is no
separate "test each element" step to write. Divisibility, sign, range
membership, "equal to any of…" — anything you can phrase as an elementwise
condition becomes a mask in one expression.


In [ ]:
import torch as t

x = t.tensor([1, 4, 6, 9, 3, 7])

# The comparison itself is the mask — same shape as x, dtype bool.
mask = x > 5
assert mask.tolist() == [False, False, True, True, False, True]

# Any elementwise condition works the same way, e.g. divisibility:
even = x % 2 == 0
assert even.tolist() == [False, True, True, False, False, False]
print("x      ", x)
print("x > 5  ", mask, mask.dtype)
print("x even ", even)




Why: one expression, no loop — the condition is written on the whole array
at once, and the result carries a True/False verdict per element.


<!-- dd:dd-q236 -->

### Problem 236 · faded — your turn

Boolean array marking entries strictly greater than a threshold.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[False,  True, False],
        [ True, False,  True]])
```


In [ ]:
import torch as t

def solve(x, threshold):
    """True exactly where x exceeds threshold."""
    return x _____ threshold


# Example run — the grader calls solve() with several different arrays
# and thresholds, including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 5.0, 2.0], [7.0, 0.5, 3.0]])
print(solve(example, 2.5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(236)


In [ ]:
#@title 💡 Solution — Problem 236
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, threshold):
    return x > threshold


example = t.tensor([[1.0, 5.0, 2.0], [7.0, 0.5, 3.0]])
print(solve(example, 2.5))


### using a mask — count and filter


Once you have a mask, two of its three uses are read-only:

- **count / reduce**: `t.count_nonzero(mask)` or `mask.sum()` (how many? —
  True behaves as 1), `mask.any()` / `mask.all()` (yes/no questions). Wrap in
  `int(...)` when a plain Python int is required.
- **filter**: `x[mask]` returns a 1-D array of just the selected elements
  (a *copy*, unlike slices) — however many there are, shape not preserved.


In [ ]:
import torch as t

x = t.tensor([1, 4, 6, 9, 3, 7])
mask = x > 5

# Count: True behaves as 1, so both spellings work.
assert t.count_nonzero(mask) == 3
assert mask.sum() == 3

# Filter: mask indexing keeps just the True positions (as a copy).
assert x[mask].tolist() == [6, 9, 7]
print("count", int(mask.sum()), "| kept", x[mask])




Why: `count_nonzero`/`sum` on a mask is the standard "how many satisfy…?";
`x[mask]` is the standard "give me those entries".


<!-- dd:dd-q52 -->

### Problem 52 · faded — your turn

Number of True entries in a boolean array, as a plain int.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
3
```


In [ ]:
import torch as t

def solve(z):
    """Count the True entries of boolean array z."""
    return int(t._____(z))


# Example run — the grader calls solve() with several arrays.
example = t.tensor([True, False, True, True])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(52)


In [ ]:
#@title 💡 Solution — Problem 52
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return int(t.count_nonzero(z))


example = t.tensor([True, False, True, True])
print(solve(example))


### combining masks and masked assignment


Masks combine with `&` (and), `|` (or), `~` (not) — NOT Python's
`and`/`or`/`not`, which fail on arrays. Because `&`/`|` bind tighter than
comparisons, each comparison needs parentheses: `(x > 3) & (x < 8)`.

The third use of a mask is **assignment**: `x[mask] = value` (or
`x[mask] *= -1`) rewrites only the selected positions, *in place*. That
mutates the original array, so the usual contract applies: "do not modify
the input" ⇒ `.copy()` first, then assign through the mask on the copy.


In [ ]:
import torch as t

x = t.tensor([1, 4, 6, 9, 3, 7])

# Combined condition + masked assignment, on a copy to protect x.
# Parentheses around EACH comparison are mandatory with & and |.
out = x.clone()
out[(out > 3) & (out < 8)] *= -1
assert out.tolist() == [1, -4, -6, 9, 3, -7]
assert x.tolist() == [1, 4, 6, 9, 3, 7]      # input untouched
print("(x > 3) & (x < 8) ->", (x > 3) & (x < 8))
print("out", out)
print("x  ", x, " <- untouched")




Why: try removing the parentheses mentally: `out > 3 & out < 8` would
evaluate `3 & out` first (bitwise on ints!) — the precedence trap is why the
parenthesized form should become muscle memory.


<!-- dd:dd-q12 -->

### Problem 12 · faded — your turn

Entries strictly between 3 and 8 negated, input untouched.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 0,  1,  2,  3, -4, -5, -6, -7,  8,  9, 10])
```


In [ ]:
import torch as t

def solve(z):
    """z with entries strictly between 3 and 8 negated (z unmodified)."""
    out = z.clone()
    out[(out > 3) _____ (out < 8)] *= -1
    return out


# Example run — the grader calls solve() with several arrays.
example = t.arange(11)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(12)


In [ ]:
#@title 💡 Solution — Problem 12
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    out = z.clone()
    out[(out > 3) & (out < 8)] *= -1
    return out


example = t.arange(11)
print(solve(example))


<!-- dd:dd-q85 -->

### Problem 85 · guided

Write a function solve(z) that takes a 2-D PyTorch integer tensor and returns a 2-D tensor containing only the rows of z that have at least one nonzero entry, in their original order. If every row is all zeros, return an empty tensor with zero rows.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[3, 0]])
```


<details>
<summary>Hints</summary>

1. Build the per-row test first: a row is dropped when EVERY entry is
   zero. That is a reduction over dim=1 producing one bool per row.
2. You want to keep the rows where that is false — negate the mask with
   `~`, then index the tensor with it.
3. `z[~(z == 0).all(dim=1)]` — an all-zero input drops to a (0, c) tensor
   by itself, which is exactly the required empty result.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return the rows of z that are not entirely zero."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([[0, 0], [3, 0], [0, 0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(85)


In [ ]:
#@title 💡 Solution — Problem 85
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z[~(z == 0).all(dim=1)]


example = t.tensor([[0, 0], [3, 0], [0, 0]])
print(solve(example))


<!-- dd:dd-q232 -->

### Problem 232 · independent

Write a function solve(x, d) that takes a 1-D PyTorch tensor of integers x and a positive integer d, and returns a boolean PyTorch tensor of the same shape. Entry i of the result must be True exactly when x[i] is divisible by d, and False otherwise. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ True, False,  True,  True, False])
```


In [ ]:
import torch as t

def solve(x, d):
    """Return a boolean array marking which elements of x are divisible by d."""
    return None


# Example run — the grader calls solve() with several different arrays and
# divisors, including edge cases. Your function must work for all of them.
example = t.tensor([3, 5, 9, 12, 14])
print(solve(example, 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(232)


In [ ]:
#@title 💡 Solution — Problem 232
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, d):
    return x % d == 0


example = t.tensor([3, 5, 9, 12, 14])
print(solve(example, 3))


<!-- dd:dd-q145 -->

### Problem 145 · independent

Write a function solve(a) that takes a 1-D float tensor and returns the indices of its LOCAL PEAKS: positions i (with 1 <= i <= len(a)-2) where a[i] is strictly greater than both immediate neighbors. The first and last positions can never qualify. Return an integer tensor (possibly empty), indices ascending.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 3])
```


In [ ]:
import torch as t

def solve(a):
    """Return indices of entries strictly greater than both neighbors."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([1.0, 3.0, 2.0, 5.0, 4.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(145)


In [ ]:
#@title 💡 Solution — Problem 145
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.where((a[1:-1] > a[:-2]) & (a[1:-1] > a[2:]))[0] + 1


print(solve(t.tensor([1.0, 3.0, 2.0, 5.0, 4.0])))


<!-- dd:dd-q202 -->

### Problem 202 · independent

Write a function solve(z) that takes a 2-D PyTorch integer tensor and returns the subarray of rows that are NOT constant — rows containing at least two different values. Rows where every entry is equal are dropped; the rest keep their order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 2, 3]])
```


In [ ]:
import torch as t

def solve(z):
    """Return the rows of z that contain at least two distinct values."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1, 1, 1], [1, 2, 3]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(202)


In [ ]:
#@title 💡 Solution — Problem 202
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    mask = (z != z[:, :1]).any(dim=1)
    return z[mask]


print(solve(t.tensor([[1, 1, 1], [1, 2, 3]])))


#### Common mistakes

- **"Combine conditions with `and`/`or`."** — Those are Python's short-circuit
  operators and raise on arrays. Masks combine with `&`, `|`, `~` — and each
  comparison must be parenthesized because `&` binds tighter than `>`.
- **"`x[mask]` keeps the array's shape."** — It returns a 1-D array of just
  the selected elements, however many there are. Only masked *assignment*
  leaves the shape intact.
- **"Counting Trues needs a loop or list.count."** — A mask is 0s and 1s:
  `mask.sum()` or `t.count_nonzero(mask)`. For yes/no rather than how-many,
  `any()`/`all()`.


<!-- dd:dd-kp-numpy-where-select -->

## Conditional values — t.where and where= arguments

`numpy.where-select`


### Choose values with t.where


Masked assignment overwrites part of an array. `t.where` instead builds a
**new array** by choosing a value at every position:

> **`t.where(condition, value_if_true, value_if_false)`**

Read it as vectorized if/else. Where the condition is `True`, PyTorch takes the
second argument. Everywhere else, it takes the third. Values may be scalars or
arrays; normal broadcasting rules apply.

For example, `t.where(z < 0, 0.0, z)` replaces negative values with zero while
leaving `z` unchanged.


> **Watch out.** `t.where(condition)` with only one argument does something different: it
returns indices of `True` positions. Choosing values always needs three
arguments.


Task: build a new array equal to `z`, except negative readings become `0.0`.


In [ ]:
import torch as t

z = t.tensor([-2.0, 0.5, 3.0, -1.0])

relu = t.where(z < 0, 0.0, z)

assert relu.tolist() == [0.0, 0.5, 3.0, 0.0]
assert z.tolist() == [-2.0, 0.5, 3.0, -1.0]
print(relu)




Why: at `z[0]`, `z[0] < 0` is `True`, so PyTorch chooses `0.0`. At
`z[1]`, the condition is `False`, so PyTorch chooses `z[1]`. Same choice runs at
every position. `t.where` returns a new array, so no `.copy()` is needed.


<!-- dd:dd-q94 -->

### Problem 94 · faded — your turn

Return `z` with `-1.0` wherever a different array, `y`, exceeds a threshold.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([-1.,  2., -1.])
```


In [ ]:
import torch as t

def solve(z, y, threshold):
    """Return z with -1.0 wherever y exceeds the threshold."""
    return t._____(_____, _____, _____)


# Example run — the grader calls solve() with several inputs.
example_z = t.tensor([1.0, 2.0, 3.0])
example_y = t.tensor([0.9, 0.1, 0.6])
print(solve(example_z, example_y, 0.5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(94)


In [ ]:
#@title 💡 Solution — Problem 94
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, y, threshold):
    out = z.clone()
    out[y > threshold] = -1.0
    return out


example_z = t.tensor([1.0, 2.0, 3.0])
example_y = t.tensor([0.9, 0.1, 0.6])
print(solve(example_z, example_y, 0.5))


### Skip unsafe positions with masked assignment


`t.where` chooses between already-computed values. Sometimes you need the
unsafe value never to be COMPUTED — division by zero being the standard case.

PyTorch has no `where=` keyword on its operators (NumPy's ufuncs do). The
torch spelling is a zeros canvas plus masked assignment, which runs the
operation only at the selected positions:

```python no-run
out = t.zeros_like(a)
nz = b != 0
out[nz] = a[nz] / b[nz]
```

`a[nz] / b[nz]` divides only the safe entries — the zero divisors are never
handed to the division at all. The canvas supplies the value left at every
skipped position, so starting from `zeros_like` makes each of them `0.0`.


> **Watch out.** `t.where(b != 0, a / b, 0.0)` does **not** prevent division by zero. Python
evaluates `a / b` first — producing `inf` or `nan` — and `t.where` only
selects afterward. Masked assignment skips the unsafe operation itself.


Task: compute `a / b`, producing `0.0` where `b` is zero without division
warnings.


In [ ]:
import torch as t

a = t.tensor([1.0, 2.0, 3.0])
b = t.tensor([2.0, 0.0, 4.0])

ratio = t.zeros_like(a)
nz = b != 0
ratio[nz] = a[nz] / b[nz]

assert ratio.tolist() == [0.5, 0.0, 0.75]
print(ratio)




Why: division runs at positions 0 and 2 only — `a[nz]` and `b[nz]` are
length-2 tensors that never contain the zero divisor. Position 1 keeps the
canvas value `0.0`. No invalid division occurs.


<!-- dd:dd-q100 -->

### Problem 100 · faded — your turn

Compute elementwise `a / b`, but return exactly `0.0` where `b` is zero.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.5000, 0.0000])
```


In [ ]:
import torch as t

def solve(a, b):
    """a / b elementwise; 0.0 where b == 0; no division by zero."""
    out = t.zeros_like(a)
    nz = _____
    out[nz] = a[nz] / b[nz]
    return out


# Example run — the grader calls solve() with several pairs.
print(solve(t.tensor([1.0, 2.0]), t.tensor([2.0, 0.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(100)


In [ ]:
#@title 💡 Solution — Problem 100
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(a, b):
    out = t.zeros_like(a)
    nz = b != 0
    out[nz] = a[nz] / b[nz]
    return out


print(solve(t.tensor([1.0, 2.0]), t.tensor([2.0, 0.0])))


<!-- dd:dd-q240 -->

### Problem 240 · guided

Write a function solve(x) that takes a 1-D PyTorch tensor of floats containing both positive and negative values, and returns a same-shape tensor in which every entry has been rounded away from zero to the nearest integer. That means positive entries move up to the next whole number, negative entries move down to the next whole number, and entries that are already whole (including zero) stay exactly as they are. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 2., -4.,  1., -2.])
```


<details>
<summary>Hints</summary>

1. Rounding away from zero is two different roundings depending on sign —
   which is a selection problem, not an arithmetic one.
2. One way avoids the branch entirely: round the MAGNITUDE up, then put
   the sign back.
3. `t.sign(x) * t.ceil(t.abs(x))` — and check zero: sign(0) is 0, so exact
   zeros stay exactly zero.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return x with every entry rounded away from zero to the nearest integer."""
    return None


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.2, -3.7, 0.5, -2.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(240)


In [ ]:
#@title 💡 Solution — Problem 240
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.sign(x) * t.ceil(t.abs(x))


example = t.tensor([1.2, -3.7, 0.5, -2.0])
print(solve(example))


<!-- dd:dd-kp-numpy-nonzero-argwhere -->

## Finding positions — nonzero and argwhere

`numpy.nonzero-argwhere`


Masks answer "*which values* satisfy the condition?". Just as often you need
"*at which positions*?" — indices, not values. Two functions return them, in
two different layouts; choosing is a matter of what you'll do with the
coordinates:

PyTorch puts both layouts behind ONE function, `t.nonzero`, and a flag picks
between them:

- **`t.nonzero(z, as_tuple=True)`** returns a **tuple of index tensors, one
  per dimension**. For 1-D: a 1-tuple holding the positions. For 2-D:
  `(rows, cols)` — two parallel tensors where `(rows[i], cols[i])` is the
  i-th hit in row-major order. This layout plugs straight back into indexing:
  `z[t.nonzero(z, as_tuple=True)]` gives the nonzero values. Because a
  boolean is just 0/1, it works on any condition:
  `t.nonzero(x > 5, as_tuple=True)`.
- **`t.nonzero(z)`** — the DEFAULT — returns **one (k, n_dims) tensor of
  coordinate rows**, for 2-D k rows of `[row, col]`. This layout is for
  *reading* coordinates (iterate them, report them, save them); it does NOT
  plug back into indexing directly. `t.argwhere(z)` is a second name for
  exactly this.

Same information, transposed packaging. Note which way round the default
falls: bare `t.nonzero` gives coordinate ROWS, so the tuple form — the one
NumPy hands you by default — is the one you have to ask for.

Once you have per-axis index arrays, whole-array geometry questions become
min/max over them — e.g. the **bounding box** of the nonzero region of a 2-D
mask is `rows.min()..rows.max()` × `cols.min()..cols.max()`.

One wrinkle for plain Python inputs: these functions accept lists, but when a
task hands you a list and wants array semantics, convert explicitly with
`t.as_tensor(x)` first — it's free for tensors and makes intent visible.


Task: find where a vector is nonzero (the tuple form), then list the
(row, col) coordinates of every nonzero cell in a 2-D mask.


In [ ]:
import torch as t

x = t.as_tensor([0, 3, 0, 0, 7, 0, -2])

# as_tuple=True returns a TUPLE of index tensors — one per dim, so 1-D gives
# a 1-tuple. This is the same structure you get from t.where(cond).
pos = t.nonzero(x, as_tuple=True)
assert isinstance(pos, tuple) and len(pos) == 1
assert pos[0].tolist() == [1, 4, 6]

# The tuple layout plugs back into indexing: the nonzero VALUES.
assert x[pos].tolist() == [3, 7, -2]

# 2-D: the DEFAULT gives coordinate ROWS — one [row, col] pair per hit,
# in row-major scan order. Made for reading, not for indexing.
# t.nonzero(z) and t.argwhere(z) are the same call.
z = t.tensor([[0, 1],
              [1, 0]])
coords = t.argwhere(z)
assert coords.tolist() == [[0, 1], [1, 0]]

# Same info as parallel per-dim tensors:
rows, cols = t.nonzero(z, as_tuple=True)
assert rows.tolist() == [0, 1] and cols.tolist() == [1, 0]

# Geometry from index arrays: bounding box of the nonzero region.
assert (rows.min(), rows.max(), cols.min(), cols.max()) == (0, 1, 0, 1)
print("as_tuple=True ->", pos, "-> values", x[pos])
print("argwhere(z) — one [row, col] per hit")
print(coords)
print("as_tuple=True on 2-D -> rows", rows, "cols", cols)




Why each step:

1. Checking `isinstance(pos, tuple)` once makes two things stick: the tuple
   layout is opt-in, and even one dimension still comes back wrapped.
   (Some drills require exactly this tuple structure.)
2. `x[pos]` closes the loop — positions in nonzero-format are *designed* to be
   used as indices.
3. The rows/cols unpacking plus min/max shows why the tuple layout wins for
   computation: each dimension's coordinates are already a vector you can
   reduce.


<!-- dd:dd-q40 -->

### Problem 40 · faded — your turn

Positions of all nonzero entries of a plain Python list, in the
tuple-of-index-tensors structure.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([1, 4, 6]),)
```


In [ ]:
import torch as t

def solve(x):
    """Return the tuple-of-index-tensors for nonzero positions of list x."""
    return t._____(t._____(x), _____=True)


# Example run — the grader calls solve() with several different lists,
# including edge cases. Your function must work for all of them.
example = [0, 3, 0, 0, 7, 0, -2]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(40)


In [ ]:
#@title 💡 Solution — Problem 40
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(x):
    return t.nonzero(t.as_tensor(x), as_tuple=True)


example = [0, 3, 0, 0, 7, 0, -2]
print(solve(example))


<!-- dd:dd-q86 -->

### Problem 86 · guided

Write a function solve(z) that takes a 2-D PyTorch integer tensor used as a mask (zeros and nonzeros) and returns a 2-D tensor of shape (k, 2) listing the (row, column) coordinates of every nonzero entry, in row-major scan order. If there are no nonzero entries, the result has zero rows.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1],
        [1, 0]])
```


<details>
<summary>Hints</summary>

1. You need a (k, 2) tensor of [row, col] coordinates in scan order — which
   spelling produces coordinate rows rather than per-dim tensors?
2. Zero rows for an all-zero mask happens automatically — an empty result is
   shape (0, 2).
3. One call, no post-processing.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return the (row, col) coordinates of every nonzero entry of z."""
    return None


# Example run — the grader calls solve() with several masks.
example = t.tensor([[0, 1], [1, 0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(86)


In [ ]:
#@title 💡 Solution — Problem 86
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.argwhere(z)


example = t.tensor([[0, 1], [1, 0]])
print(solve(example))


<!-- dd:dd-q110 -->

### Problem 110 · independent

Write a function solve(z) that takes a 2-D PyTorch tensor containing at least one nonzero entry and returns a tuple of four plain ints (row_min, row_max, col_min, col_max): the tightest bounding box of the nonzero entries — the first and last row indices that contain any nonzero value, and likewise for columns (both bounds inclusive).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(1, 2, 2, 2)
```


In [ ]:
import torch as t

def solve(z):
    """Return (row_min, row_max, col_min, col_max) of z's nonzero entries."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.zeros((4, 4), dtype=t.int64)
example[1:3, 2] = 1
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(110)


In [ ]:
#@title 💡 Solution — Problem 110
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    rows = t.any(z, dim=1)
    cols = t.any(z, dim=0)
    rmin, rmax = t.where(rows)[0][[0, -1]]
    cmin, cmax = t.where(cols)[0][[0, -1]]
    return int(rmin), int(rmax), int(cmin), int(cmax)


example = t.zeros((4, 4), dtype=t.int64)
example[1:3, 2] = 1
print(solve(example))


#### Common mistakes

- **"`t.nonzero` behaves like NumPy's."** — It does not. NumPy's default is
  the per-axis tuple; PyTorch's default is coordinate rows (what NumPy calls
  argwhere). Passing `as_tuple=True` is what recovers the NumPy layout, and
  code that indexes with the default output breaks.
- **"For a 1-D tensor, as_tuple=True returns the indices directly."** — It
  returns a 1-TUPLE containing the index tensor. Unpack with `pos[0]` or
  `idx, = t.nonzero(x, as_tuple=True)` when you want the bare tensor.
- **"I need a loop to find positions matching a condition."** —
  `t.nonzero(condition, as_tuple=True)` does it: a mask is already the 0/1
  tensor nonzero scans.


<!-- dd:dd-kp-numpy-argmin-argmax -->

## Locating extremes — argmin and argmax

`numpy.argmin-argmax`


### argmin/argmax — the index, not the value


`min`/`max` tell you the extreme **value**; the `arg` twins tell you **where
it lives**:

- **`t.argmin(v)` / `v.argmin()`** — index of the smallest element.
- **`t.argmax(v)` / `v.argmax()`** — index of the largest.

**Ties break to the first occurrence.** If the extreme value appears more
than once, you get the smallest index — deterministically. Many drills state
"replace only the first occurrence"; argmin/argmax gives exactly that for
free.

Like other reductions, the result is a 0-dim TENSOR, not a Python int;
wrap in `int(...)` when
a plain Python int is required. (Per-row/column argmax with `axis=` appears
in the broadcasting lesson — same idea, one axis at a time.)


In [ ]:
import torch as t

v = t.tensor([4.0, 2.0, 7.0, 2.0, 9.0])

# Index of the minimum. 2.0 appears twice — argmin reports the FIRST.
i = int(t.argmin(v))
assert i == 1

# The value at that index is the min itself.
assert v[i] == v.min()
print("v", v)
print("argmin ->", i, "| v[i] =", v[i].item(), "| v.min() =", v.min().item())




Why: `int(...)` at the boundary again — graders asking for "a plain Python
int" reject `t.int64`. And `v[v.argmax()]` is how you get the value back
when you need both.


<!-- dd:dd-q38 -->

### Problem 38 · faded — your turn

Index of the smallest element (first occurrence on ties), as a plain int.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
1
```


In [ ]:
import torch as t

def solve(v):
    """Index of v's smallest element, first occurrence on ties."""
    return int(t._____(v))


# Example run — the grader calls solve() with several different vectors,
# including edge cases. Your function must work for all of them.
example = t.tensor([4.0, 2.0, 7.0, 2.5, 9.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(38)


In [ ]:
#@title 💡 Solution — Problem 38
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v):
    return int(t.argmin(v))


example = t.tensor([4.0, 2.0, 7.0, 2.5, 9.0])
print(solve(example))


### the index as a handle for surgery


"Replace the largest entry with 0" is: copy (if the input must survive), then
`out[out.argmax()] = 0`. One read, one write, no scanning loop — the index is
a *handle* you use to edit the array.

The habit to build: **protect the input, then operate.** Copy first, then
assign through the index. It reads cleanest and never backfires.


In [ ]:
import torch as t

v = t.tensor([4.0, 2.0, 7.0, 2.0, 9.0])

# Replace the max with 0 on a copy: the index is the handle.
out = v.clone()
out[out.argmax()] = 0.0          # argmax -> 4; out[4] = 0
assert out.tolist() == [4.0, 2.0, 7.0, 2.0, 0.0]
assert v.tolist() == [4.0, 2.0, 7.0, 2.0, 9.0]   # input intact
print("out", out)
print("v  ", v, " <- untouched")




Why: the copy-then-assign order matters when the input must survive —
`v[v.argmax()] = 0` would mutate the caller's array.


<!-- dd:dd-q219 -->

### Problem 219 · faded — your turn

Largest entry replaced with 0 (first occurrence only), input untouched.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 3., -1.,  0.,  2.])
```


In [ ]:
import torch as t

def solve(x):
    """x with its largest entry replaced by 0.0, without mutating x."""
    result = x._____()
    result[t._____(x)] = 0.0
    return result


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([3.0, -1.0, 7.5, 2.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(219)


In [ ]:
#@title 💡 Solution — Problem 219
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    result = x.clone()
    result[t.argmax(x)] = 0.0
    return result


example = t.tensor([3.0, -1.0, 7.5, 2.0])
print(solve(example))


### closest-to-target — argmin on a transformed array


argmin/argmax compose with transformed arrays. The pattern
`t.argmin(t.abs(z - target))` answers "which entry is *closest to*
target?" — build the quantity you want minimized, then ask where its minimum
sits. Any "closest / best / most-similar" task is this pattern with a
different transform.

Keep the roles straight: the *transformed* array chooses the index; the
*original* array supplies the value at that index.


In [ ]:
import torch as t

v = t.tensor([4.0, 2.0, 7.0, 2.0, 9.0])

# "Closest to target" = argmin of a transformed array.
target = 6.5
j = int(t.argmin(t.abs(v - target)))
assert j == 2                     # |7.0 - 6.5| = 0.5 is the smallest gap
closest_value = v[j]
assert closest_value == 7.0
print("gaps to", target, ":", t.abs(v - target))
print("smallest gap at index", j, "-> value", closest_value.item())




Why: no sorting needed — sorting is O(n log n) and loses positions;
`argmin(|v - t|)` is one pass and keeps them.


<!-- dd:dd-q98 -->

### Problem 98 · faded — your turn

The INDEX of the entry closest to target, as a plain int.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
1
```


In [ ]:
import torch as t

def solve(z, target):
    """Index of the entry of z closest to target."""
    return int(t._____(t._____(z - target)))


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([0.1, 0.4, 0.8]), 0.5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(98)


In [ ]:
#@title 💡 Solution — Problem 98
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, target):
    return int(t.abs(z - target).argmin())


print(solve(t.tensor([0.1, 0.4, 0.8]), 0.5))


<!-- dd:dd-q1 -->

### Problem 1 · guided

Write a function solve(z) that takes a 2-D PyTorch tensor z and returns a 1-D integer tensor containing, for each row, the index of that row's largest value. If a row's maximum appears more than once, report the first (leftmost) occurrence. The matrix can have any shape, including a single row or a single column.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 0])
```


<details>
<summary>Hints</summary>

1. Per row means the reduction runs ALONG the columns — which axis number
   is that for a 2-D tensor?
2. You want the position of the max, not the max itself, and torch already
   breaks ties leftmost.
3. `z.argmax(dim=1)` — first-occurrence tie-breaking is the documented
   default, so no extra work is needed.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return the per-row index of the largest value in z."""
    return None


# Example run — the grader calls solve() with several different arrays.
example = t.tensor([[1, 9, 3], [7, 2, 5]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1)


In [ ]:
#@title 💡 Solution — Problem 1
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.argmax(dim=1)


example = t.tensor([[1, 9, 3], [7, 2, 5]])
print(solve(example))


<!-- dd:dd-q24 -->

### Problem 24 · independent

Write a function solve(z) that takes a 1-D PyTorch tensor of floats and returns a new tensor in which the largest value has been replaced by 0.0, with everything else unchanged. If the maximum occurs more than once, replace only its first occurrence. Do not modify the input tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2., 0., 4.])
```


In [ ]:
import torch as t

def solve(z):
    """Return z with its largest value replaced by 0.0."""
    return None


# Example run — the grader calls solve() with several vectors.
example = t.tensor([2.0, 9.0, 4.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(24)


In [ ]:
#@title 💡 Solution — Problem 24
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    out = z.clone()
    out[out.argmax()] = 0.0
    return out


example = t.tensor([2.0, 9.0, 4.0])
print(solve(example))


<!-- dd:dd-q61 -->

### Problem 61 · independent

Write a function solve(z, v) that takes a 1-D PyTorch numeric tensor z and a scalar v, and returns the single element of z that is closest to v (minimizing the absolute difference). If two elements are equally close, return the one at the lower index. Return a scalar value, not an tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(10)
```


In [ ]:
import torch as t

def solve(z, v):
    """Return the element of z nearest to the scalar v."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.tensor([0, 10, 20, 30])
print(solve(example, 12.4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(61)


In [ ]:
#@title 💡 Solution — Problem 61
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, v):
    return z[t.abs(z - v).argmin()]


example = t.tensor([0, 10, 20, 30])
print(solve(example, 12.4))


<!-- dd:dd-q168 -->

### Problem 168 · independent

Write a function solve(z) that takes a 2-D PyTorch integer tensor and returns, for each row, the column index of the row's largest value — but breaking ties by the LAST occurrence instead of argmax's usual first-occurrence rule.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2, 2])
```


In [ ]:
import torch as t

def solve(z):
    """Return each row's argmax with ties broken by the LAST occurrence."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1, 5, 5], [7, 2, 7]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(168)


In [ ]:
#@title 💡 Solution — Problem 168
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    rev = z.flip(1)
    return z.shape[1] - 1 - rev.argmax(dim=1)


print(solve(t.tensor([[1, 5, 5], [7, 2, 7]])))


#### Common mistakes

- **"argmax returns the maximum."** — It returns the *index* of the maximum.
  `v[v.argmax()]` is the value; needing both is common and costs one extra
  read.
- **"Ties are an error / unspecified."** — Ties resolve to the first
  occurrence, deterministically. Tasks that say "first occurrence" are
  describing argmin/argmax's default, not asking for extra work.
- **"Closest-to-target needs sorting."** — Sorting is O(n log n) and loses
  positions; `argmin(|v - t|)` is one pass and keeps them. Save sorting for
  when you need full order, not one winner.


<!-- dd:dd-kp-numpy-fancy-indexing -->

## Fancy indexing — index arrays and lookup tables

`numpy.fancy-indexing`


### index with an array — select and reorder at once


Slices select *regular* pieces — ranges with a stride. **Fancy indexing**
lifts that restriction: index with an **array of integers**, and you get the
elements at exactly those positions, *in the order you listed them,
repetitions allowed*:

```python no-run
x[t.tensor([3, 0, 0, 2])]   # elements 3, 0, 0, 2 — any order, any repeats
```

**Reordering is indexing.** Rows of a matrix in a new order: `z[perm]` where
`perm` is a permutation of row indices — row i of the result is row
`perm[i]` of `z`. The index array describes *where results come from*, not
where they go. (On 2-D arrays a single index array selects whole ROWS;
columns are `z[:, idx]`.) Negative indices work just like plain indexing —
and unlike a slice, fancy indexing always returns a **copy**.


In [ ]:
import torch as t

z = t.arange(6).reshape(3, 2)      # [[0,1],[2,3],[4,5]]

# Rows in the order 2, 0, 1. Read it as: "give me row 2, then row 0,
# then row 1" — the index array IS the new row order.
perm = t.tensor([2, 0, 1])
reordered = z[perm]
assert reordered.tolist() == [[4, 5], [0, 1], [2, 3]]

# Fancy indexing copies — mutating the result leaves z alone.
reordered[0, 0] = 99
assert z[2, 0] == 4
print("order", perm)
print(reordered, " <- 99 written here")
print("z unchanged:", z.tolist())




Why: verbalizing `z[perm]` ("row perm[i] lands at position i") resolves the
direction confusion that otherwise haunts permutation tasks.


<!-- dd:dd-q139 -->

### Problem 139 · faded — your turn

Exactly the rows named by idx, in idx's order (repeats and negatives legal).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[50., 60.],
        [10., 20.]])
```


In [ ]:
import torch as t

def solve(x, idx):
    """Rows of x selected and ordered by idx."""
    return x[_____]


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([[10.0, 20.0], [30.0, 40.0], [50.0, 60.0]])
example_idx = t.tensor([2, 0])
print(solve(example, example_idx))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(139)


In [ ]:
#@title 💡 Solution — Problem 139
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, idx):
    return x[idx]


example = t.tensor([[10.0, 20.0], [30.0, 40.0], [50.0, 60.0]])
example_idx = t.tensor([2, 0])
print(solve(example, example_idx))


### a lookup table is indexing


If `values` is a table of length K and `labels` holds class ids 0..K-1, then
`values[labels]` replaces every label with its looked-up value — the output
is shaped like `labels`; the indexed table just supplies the entries. Any
"map each id to its value" task is one indexing expression.

Note which array is inside the brackets: the LABELS index, the TABLE is
indexed. Output shape always follows the indexer.


In [ ]:
import torch as t

# Lookup table: labels index into values. Output has labels' shape.
values = t.tensor([10, 20, 30])
labels = t.tensor([0, 2, 1, 2, 0])
decoded = values[labels]
assert decoded.tolist() == [10, 30, 20, 30, 10]
print("labels ", labels, "shape", tuple(labels.shape))
print("decoded", decoded, "shape", tuple(decoded.shape), "<- labels' shape")




Why: this is a single vectorized gather in C — if you're writing
`t.tensor([values[l] for l in labels])`, replace it.


<!-- dd:dd-q71 -->

### Problem 71 · faded — your turn

Replace every label by its value from a lookup table.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([10, 30, 20, 30, 10])
```


In [ ]:
import torch as t

def solve(labels, values):
    """values[labels[i]] for every i — as one indexing expression."""
    return _____[_____]


# Example run — the grader calls solve() with several inputs.
example_labels = t.tensor([0, 2, 1, 2, 0])
example_values = t.tensor([10, 20, 30])
print(solve(example_labels, example_values))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(71)


In [ ]:
#@title 💡 Solution — Problem 71
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(labels, values):
    return values[labels]


example_labels = t.tensor([0, 2, 1, 2, 0])
example_values = t.tensor([10, 20, 30])
print(solve(example_labels, example_values))


### swaps are simultaneous


`out[[0, -1]] = out[[-1, 0]]` exchanges first and last rows in one
statement: the right side is gathered *before* the left side is written, so
nothing is clobbered mid-swap. The swap works *because* fancy indexing
copies on read.

Contrast: the pure-Python idiom `z[0], z[-1] = z[-1], z[0]` is NOT safe on
arrays — each `z[i]` is a view, so the first assignment overwrites data the
second still needs.


In [ ]:
import torch as t

# Simultaneous swap of first and last rows — RHS gathered before write.
w = t.tensor([[1, 2], [3, 4], [5, 6]])
w[[0, -1]] = w[[-1, 0]]
assert w.tolist() == [[5, 6], [3, 4], [1, 2]]
print(w)




Why: the list form `[[0, -1]]` is fancy indexing with a 2-element index
tensor — by the time PyTorch writes `w[[0, -1]]`, the old rows are already
safely gathered.


<!-- dd:dd-q25 -->

### Problem 25 · faded — your turn

First and last rows exchanged (new array, input unmodified).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[7, 8, 9],
        [4, 5, 6],
        [1, 2, 3]])
```


In [ ]:
import torch as t

def solve(x):
    """x with its first and last rows swapped, x unmodified."""
    out = x.clone()
    out[[0, -1]] = out[_____]
    return out


# Example run — the grader calls solve() with several different matrices,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(25)


In [ ]:
#@title 💡 Solution — Problem 25
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    out = x.clone()
    out[[0, -1]] = out[[-1, 0]]
    return out


example = t.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(solve(example))


<!-- dd:dd-q136 -->

### Problem 136 · guided

Write a function solve(z, i, j) that takes a 2-D PyTorch tensor and two row indices, and returns a new tensor with rows i and j exchanged and everything else unchanged. The input tensor must not be modified. i and j may be equal.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[3, 4, 5],
        [0, 1, 2],
        [6, 7, 8]])
```


<details>
<summary>Hints</summary>

1. Row swap is a fancy-index assignment: read the two rows in one order,
   write them back in the other.
2. Work on a clone — the drill checks the input is untouched — and
   remember i may equal j, which must still be a no-op.
3. `out = z.clone()`, then `out[[i, j]] = out[[j, i]]`. The right-hand
   side is materialized before the write, so no aliasing problem.

</details>


In [ ]:
import torch as t

def solve(z, i, j):
    """Return z with rows i and j swapped."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.arange(9).reshape(3, 3)
print(solve(example, 0, 1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(136)


In [ ]:
#@title 💡 Solution — Problem 136
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, i, j):
    out = z.clone()
    out[[i, j]] = out[[j, i]]
    return out


example = t.arange(9).reshape(3, 3)
print(solve(example, 0, 1))


<!-- dd:dd-q70 -->

### Problem 70 · independent

Write a function solve(z, perm) that takes a 2-D PyTorch tensor z and a 1-D integer tensor perm holding a permutation of the row indices of z. Return a new tensor whose i-th row is row perm[i] of z — the rows of z reordered according to perm.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[4, 5],
        [0, 1],
        [2, 3]])
```


In [ ]:
import torch as t

def solve(z, perm):
    """Return the rows of z reordered by the permutation perm."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.arange(6).reshape(3, 2)
print(solve(example, t.tensor([2, 0, 1])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(70)


In [ ]:
#@title 💡 Solution — Problem 70
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, perm):
    return z[perm]


example = t.arange(6).reshape(3, 2)
print(solve(example, t.tensor([2, 0, 1])))


<!-- dd:dd-q30 -->

### Problem 30 · independent

Write a function solve(x) that takes a 2-D PyTorch integer tensor x and returns a new tensor of the same shape in which the first and last columns have exchanged places. Every other column, and the order of the rows, must be unchanged. Do not modify x itself — the grader checks that the original tensor is left intact. The matrix can be any shape, including a single row or a single column.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2, 1, 0],
        [5, 4, 3],
        [8, 7, 6]])
```


In [ ]:
import torch as t

def solve(x):
    """Return a copy of the 2-D array x with its first and last columns exchanged."""
    return None


# Example run — the grader calls solve() with several different matrices,
# including edge cases. Your function must work for all of them.
example = t.arange(9).reshape(3, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(30)


In [ ]:
#@title 💡 Solution — Problem 30
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    out = x.clone()
    out[:, [0, -1]] = out[:, [-1, 0]]
    return out


example = t.arange(9).reshape(3, 3)
print(solve(example))


<!-- dd:dd-q101 -->

### Problem 101 · independent

Write a function solve(z, k, rng) that takes a 2-D PyTorch tensor z with all-distinct rows, an integer k, and a PyTorch random Generator rng. Return a 2-D tensor of k rows sampled from z uniformly at random WITHOUT replacement using rng. The grader checks the shape, that every returned row exists in z, and that the k sampled rows are all different — any valid sample passes.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [3, 4, 5]])
```


In [ ]:
import torch as t

def solve(z, k, rng):
    """Return k distinct rows of z sampled at random without replacement."""
    return None


# Example run — the grader calls solve() with seeded generators.
example = t.arange(12).reshape(4, 3)
print(solve(example, 2, t.Generator().manual_seed(0)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(101)


In [ ]:
#@title 💡 Solution — Problem 101
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z, k, gen):
    idx = t.randperm(z.shape[0], generator=gen)[:k]
    return z[idx]


example = t.arange(12).reshape(4, 3)
print(solve(example, 2, t.Generator().manual_seed(0)))


<!-- dd:dd-q177 -->

### Problem 177 · independent

Write a function solve(a, mapping) that takes a 1-D non-negative integer tensor a and a Python dict of {old_value: new_value} integer pairs whose keys include every value appearing in a. Return a new tensor with each entry replaced by its mapped value, WITHOUT looping over the elements of a (looping over the small dict is fine).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([8, 7, 6, 7, 8, 9])
```


In [ ]:
import torch as t

def solve(a, mapping):
    """Return a with each value replaced via the mapping dict."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([1, 2, 3, 2, 1, 0]), {0: 9, 1: 8, 2: 7, 3: 6}))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(177)


In [ ]:
#@title 💡 Solution — Problem 177
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, mapping):
    table = t.arange(max(mapping) + 1)
    for k, v in mapping.items():
        table[k] = v
    return table[a]


print(solve(t.tensor([1, 2, 3, 2, 1, 0]), {0: 9, 1: 8, 2: 7, 3: 6}))


<!-- dd:dd-q184 -->

### Problem 184 · independent

Write a function solve(z, i, j, v) that takes a SYMMETRIC 2-D PyTorch tensor z, two indices, and a value, and returns a new tensor equal to z except that BOTH z[i, j] and z[j, i] are set to v — preserving symmetry. The input must not be modified. i and j may be equal (the diagonal).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 0., 5.],
        [0., 0., 0.],
        [5., 0., 0.]])
```


In [ ]:
import torch as t

def solve(z, i, j, v):
    """Return z with [i,j] and [j,i] both set to v (symmetry kept)."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.zeros((3, 3)), 0, 2, 5.0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(184)


In [ ]:
#@title 💡 Solution — Problem 184
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, i, j, v):
    out = z.clone()
    out[i, j] = v
    out[j, i] = v
    return out


print(solve(t.zeros((3, 3)), 0, 2, 5.0))


#### Common mistakes

- **"Fancy indexing returns a view like slicing."** — It returns a COPY.
  Consequence: `z[idx][0] = 5` modifies the copy and silently discards it;
  write through the original (`z[idx_of_target] = 5`) when mutation is the
  goal.
- **"Swapping rows needs a temporary."** — `z[[i, j]] = z[[j, i]]` is safe:
  the right side is fully read before the write. (The pure-Python idiom
  `z[i], z[j] = z[j], z[i]` is NOT safe on arrays — the first assignment
  overwrites data the second still needs, because each `z[i]` is a view.)
- **"values[labels] loops over labels under the hood in Python."** — It's a
  single vectorized gather in C. If you're writing
  `t.tensor([values[l] for l in labels])`, replace it.


<!-- dd:dd-kp-numpy-unique -->

## Distinct values — t.unique and friends

`numpy.unique`


**`t.unique(z)`** returns the distinct values of an array, **sorted
ascending** — deduplication and ordering in one call. Its real depth is in
the optional outputs, each answering a different question about the
duplicates it collapsed:

- **`return_counts=True`** — how many times does each distinct value occur?
  Returns `(values, counts)`, aligned by position. This is the histogram of
  the data's actual values.
- **`return_index=True`** — where did each distinct value FIRST appear?
  Returns indices into the original array (handy for order-of-first-appearance
  reconstructions).
- **`return_inverse=True`** — for each original element, which distinct value
  is it? (`values[inverse]` rebuilds the input — a factorization into
  vocabulary + codes.)

Two generalizations worth knowing now:

- **Rows as units**: `t.unique(z, dim=0)` deduplicates whole ROWS of a 2-D
  tensor (sorted lexicographically — first column, then second…). Without
  `dim=`, a 2-D input is flattened and you get distinct *scalars*.
- **Set operations between tensors**: PyTorch ships only the membership test,
  **`t.isin(a, b)`** — a boolean mask over `a`. There is no `intersect1d`,
  `union1d` or `setdiff1d`, and you do not need them: composing `unique` with
  `isin` covers the family.

  - intersection: `ua = t.unique(a); ua[t.isin(ua, b)]`
  - difference:   `ua = t.unique(a); ua[~t.isin(ua, b)]`
  - union:        `t.unique(t.cat([a, b]))`

The mental model: `unique` hands back a canonical (sorted, deduplicated)
form, and `isin` filters it — if a task says "each value exactly once,
ascending", it is describing this family.


Task: get the vocabulary of a measurement vector with occurrence counts, and
find which values two arrays share.


In [ ]:
import torch as t

z = t.tensor([3, 1, 2, 3, 1, 3])

# Distinct values, sorted — and their aligned counts.
values, counts = t.unique(z, return_counts=True)
assert values.tolist() == [1, 2, 3]
assert counts.tolist() == [2, 1, 3]      # counts[i] belongs to values[i]

# The pair answers "most common value" without a Python Counter:
assert values[counts.argmax()] == 3

# return_inverse: codes that rebuild the input from the vocabulary.
values2, inverse = t.unique(z, return_inverse=True)
assert values2[inverse].tolist() == z.tolist()

# Set intersection of two tensors: shared values, once each, ascending.
# unique gives the sorted vocabulary; isin filters it down to the shared part.
a = t.tensor([4, 1, 3, 2, 3])
b = t.tensor([3, 4, 4, 8, 0])
ua = t.unique(a)
assert ua[t.isin(ua, b)].tolist() == [3, 4]
print("z       ", z)
print("values  ", values, "counts", counts)
print("most common:", values[counts.argmax()].item())
print("inverse ", inverse, "-> rebuilt", values2[inverse])
print("intersection of", a.tolist(), "and", b.tolist(), ":",
      ua[t.isin(ua, b)])




Why each step:

1. `values`/`counts` alignment is positional — index i of each refers to the
   same distinct value. Downstream questions (most common, rarest, values
   occurring exactly once) are reductions over `counts` followed by indexing
   into `values`.
2. `values[counts.argmax()]` chains last KP's argmax with this KP's aligned
   arrays — this composition is the standard "mode" idiom for small unique
   sets.
3. `intersect1d` returning `[3, 4]` (not `[4, 3]`, not `[3, 4, 4]`)
   demonstrates the canonical-form contract: sorted, each value once —
   regardless of input order or multiplicity.


<!-- dd:dd-q14 -->

### Problem 14 · faded — your turn

Distinct values ascending, with aligned occurrence counts.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([1, 2, 3]), tensor([1, 2, 3]))
```


In [ ]:
import torch as t

def solve(z):
    """(distinct values ascending, counts aligned with them)."""
    return t._____(z, _____=True)


# Example run — the grader calls solve() with several arrays.
example = t.tensor([1, 2, 2, 3, 3, 3])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(14)


In [ ]:
#@title 💡 Solution — Problem 14
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.unique(z, return_counts=True)


example = t.tensor([1, 2, 2, 3, 3, 3])
print(solve(example))


<!-- dd:dd-q241 -->

### Problem 241 · guided

Write a function solve(a, b) that takes two 1-D PyTorch integer tensors and returns a 1-D tensor of the values that appear in both a and b. Each shared value must appear exactly once in the result, and the result must be sorted in ascending order. If the two tensors have no values in common, return an empty 1-D tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([3, 4])
```


<details>
<summary>Hints</summary>

1. Values appearing in BOTH arrays, each exactly once, ascending — that
   sentence is the contract of one set function.
2. The `*1d` family: intersect, union, setdiff. Which one?
3. Empty intersection falls out naturally as an empty array — no special
   case needed.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return the sorted 1-D array of unique values present in both a and b."""
    return None


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
a = t.tensor([4, 1, 3, 2, 3])
b = t.tensor([3, 4, 4, 8, 0])
print(solve(a, b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(241)


In [ ]:
#@title 💡 Solution — Problem 241
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(a, b):
    ua = t.unique(a)
    return ua[t.isin(ua, b)]


a = t.tensor([4, 1, 3, 2, 3])
b = t.tensor([3, 4, 4, 8, 0])
print(solve(a, b))


<!-- dd:dd-q102 -->

### Problem 102 · independent

Write a function solve(z) that takes a 1-D PyTorch integer tensor and returns a tuple (values, counts): the distinct values of z and their occurrence counts, both ordered by DESCENDING count (most frequent value first). You may assume all counts are distinct, so the order is unambiguous.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([4, 7, 1]), tensor([3, 2, 1]))
```


In [ ]:
import torch as t

def solve(z):
    """Return (values, counts) sorted by descending occurrence count."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([4, 4, 4, 7, 7, 1])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(102)


In [ ]:
#@title 💡 Solution — Problem 102
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    vals, cnts = t.unique(z, return_counts=True)
    order = t.argsort(cnts).flip(0)
    return vals[order], cnts[order]


print(solve(t.tensor([4, 4, 4, 7, 7, 1])))


<!-- dd:dd-q148 -->

### Problem 148 · independent

Write a function solve(a) that takes a 1-D PyTorch integer tensor and returns the indices of the FIRST occurrence of each distinct value, in ascending index order — the positions where a previously unseen value appears as you scan left to right.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0, 1, 3, 5])
```


In [ ]:
import torch as t

def solve(a):
    """Return ascending indices of each value's first appearance in a."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([4, 2, 4, 1, 2, 5])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(148)


In [ ]:
#@title 💡 Solution — Problem 148
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(a):
    vals, inverse = t.unique(a, return_inverse=True)
    first = t.full((vals.numel(),), a.numel(), dtype=t.int64)
    first.scatter_reduce_(0, inverse, t.arange(a.numel()), reduce="amin")
    return t.sort(first).values


print(solve(t.tensor([4, 2, 4, 1, 2, 5])))


<!-- dd:dd-q79 -->

### Problem 79 · independent

Write a function solve(z) that takes a 2-D PyTorch tensor and returns a 2-D tensor containing each distinct ROW of z exactly once, ordered lexicographically (compare first column, then second, and so on). Duplicate rows are collapsed to a single occurrence.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 9],
        [1, 2]])
```


In [ ]:
import torch as t

def solve(z):
    """Return the unique rows of z, lexicographically sorted."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([[1, 2], [1, 2], [0, 9]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(79)


In [ ]:
#@title 💡 Solution — Problem 79
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.unique(z, dim=0)


example = t.tensor([[1, 2], [1, 2], [0, 9]])
print(solve(example))


#### Common mistakes

- **"unique preserves input order."** — It sorts. If you need
  order-of-first-appearance, combine `return_index=True` with a sort of those
  indices — the sorted-values default is the contract, not a coincidence.
- **"unique on a matrix gives unique rows."** — Without `axis=0` the array is
  flattened first and you get distinct scalars. Row-level deduplication is
  explicitly `t.unique(z, axis=0)`.
- **"Intersection = loop with `in`."** — `t.unique(a)` filtered by
  `t.isin(...)` (or `t.isin(a, b)` alone when you want a mask on `a`). The
  Python-level loop is quadratic and unvectorized.


<!-- dd:dd-kp-numpy-diag-triangles -->

## Diagonals, triangles, and trace

`numpy.diag-triangles`


### extracting a diagonal with t.diag


Give `t.diag` a **2-D array** and it *extracts* a diagonal as a 1-D array.
Which diagonal is picked by one convention you'll reuse everywhere: the
**offset k**.

- `k = 0` (the default) is the **main diagonal** — top-left to bottom-right.
- **Positive k** counts diagonals *above* the main one: `t.diag(z, k=1)` is
  the superdiagonal.
- **Negative k** counts diagonals *below*: `t.diag(z, k=-1)` is the
  subdiagonal.

Every function in this family (`diag`, `trace`, `triu`, `tril`, `t.eye`'s
`k=`) shares this exact convention — learn it once here.


Task: extract the main, upper, and lower diagonals of a 3×3 matrix.


In [ ]:
import torch as t

z = t.arange(1, 10).reshape(3, 3)   # [[1,2,3],[4,5,6],[7,8,9]]

# The offset convention in action: up is positive, down is negative.
assert t.diag(z).tolist() == [1, 5, 9]        # k=0, main
assert t.diag(z, diagonal=1).tolist() == [2, 6]      # one above
assert t.diag(z, diagonal=-1).tolist() == [4, 8]     # one below
print(z)
print("k= 0", t.diag(z))
print("k= 1", t.diag(z, diagonal=1))
print("k=-1", t.diag(z, diagonal=-1))




Why: `t.arange(1, 10).reshape(3, 3)` is the perfect test matrix — every
entry announces its own position, so you can *see* which diagonal came out.
When unsure about a k, test on it.


<!-- dd:dd-q77 -->

### Problem 77 · faded — your turn

The k-th diagonal of a square matrix, using the standard sign convention.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 1,  7, 13, 19])
```


In [ ]:
import torch as t

def solve(z, k):
    """The k-th diagonal of z (k=0 main, positive above, negative below)."""
    return t._____(z, _____)


# Example run — the grader calls solve() with several inputs.
example = t.arange(25).reshape(5, 5)
print(solve(example, 1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(77)


In [ ]:
#@title 💡 Solution — Problem 77
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, k):
    return t.diag(z, diagonal=k)


example = t.arange(25).reshape(5, 5)
print(solve(example, 1))


### t.trace — sum of the main diagonal


`t.trace(z)` is the **sum of the main diagonal**, returned as a scalar. It's
equivalent to `t.diag(z).sum()` — extract, then reduce — collapsed into one
call. It also accepts `offset=` for other diagonals, with the same sign
convention as `t.diag`'s k.


In [ ]:
import torch as t

z = t.arange(1, 10).reshape(3, 3)

# Trace = main diagonal summed: 1 + 5 + 9.
assert t.trace(z) == 15

# Same thing spelled as extract-then-reduce:
assert t.diag(z).sum() == 15

# t.trace has NO offset argument — for any other diagonal, compose:
assert t.diagonal(z, offset=1).sum() == 8
print("trace       ", int(t.trace(z)))
print("diag().sum()", int(t.diag(z).sum()))
print("offset=1 sum", int(t.diagonal(z, offset=1).sum()))




Why: "sum of the k-th diagonal" is the composition habit this whole toolkit
builds — *structure function, then reduce*. `t.trace` is just the shortcut
for the most common case.


<!-- dd:dd-q237 -->

### Problem 237 · faded — your turn

The trace of a square matrix, as a scalar.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(5.)
```


In [ ]:
import torch as t

def solve(z):
    """Sum of z's main-diagonal entries."""
    return t._____(z)


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(237)


In [ ]:
#@title 💡 Solution — Problem 237
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.trace(z)


example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


### building a diagonal matrix with t.diag


`t.diag` is a **two-way street** — its behavior depends on the input's rank.
You've seen the 2-D direction (extract). Give it a **1-D array** and it
*builds*: a square matrix with those values ON the diagonal, zeros elsewhere.
`t.diag(t.tensor([1, 2, 3]))` is a 3×3 diagonal matrix (and unlike NumPy it
insists on a tensor, not a bare list); `t.diag(vals, diagonal=-1)` places
them just below the main diagonal (the result grows to fit: length n on
offset k gives shape (n+|k|, n+|k|)).

Build-vs-extract is decided by the INPUT's rank, not by an argument — passing
a vector where you meant a matrix silently switches modes, so always know
which one your data is.


In [ ]:
import torch as t

# BUILD (1-D input): same function, other direction.
d = t.diag(t.tensor([1.0, 2.0, 3.0]))
assert d.tolist() == [[1.0, 0.0, 0.0],
                      [0.0, 2.0, 0.0],
                      [0.0, 0.0, 3.0]]

# With an offset, the matrix grows to fit: 2 values on k=-1 -> 3x3.
assert t.diag(t.tensor([7, 8]), diagonal=-1).tolist() == [[0, 0, 0],
                                          [7, 0, 0],
                                          [0, 8, 0]]
print(d)
print("two values on k=-1 grow a 3x3:")
print(t.diag(t.tensor([7, 8]), diagonal=-1))




Why: the offset build is how you make shift/step matrices (e.g. the values
1..n-1 just below the diagonal) in one call — no loops, no indexing.


<!-- dd:dd-q47 -->

### Problem 47 · faded — your turn

A k×k matrix with the given values on its main diagonal.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 0., 0.],
        [0., 2., 0.],
        [0., 0., 3.]])
```


In [ ]:
import torch as t

def solve(vals):
    """Square matrix with vals on the diagonal, zeros elsewhere."""
    return t._____(t.as_tensor(vals))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.0, 2.0, 3.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(47)


In [ ]:
#@title 💡 Solution — Problem 47
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(vals):
    return t.diag(t.as_tensor(vals))


example = t.tensor([1.0, 2.0, 3.0])
print(solve(example))


### triangles — t.triu and t.tril


`t.triu(z, k=0)` keeps the **upper triangle** — everything ON and ABOVE
diagonal k — and zeroes the rest; `t.tril(z, k=0)` keeps the lower. The `k`
shifts the cut line, with the same sign convention as always:
`t.tril(z, k=-1)` keeps only *strictly below* the main diagonal.

To build a triangular matrix from scratch, compose with a constructor:
`t.triu(t.ones((n, n)))` is the upper-triangular matrix of ones — pass
`dtype=bool` to `ones` and it's a boolean mask instead. Constructor →
structure function is the standard recipe.


In [ ]:
import torch as t

z = t.arange(1, 10).reshape(3, 3)

# Keep upper (with diagonal), zero the rest...
assert t.triu(z).tolist() == [[1, 2, 3],
                               [0, 5, 6],
                               [0, 0, 9]]

# ...or keep ONLY the strictly-below part by shifting the cut to k=-1.
assert t.tril(z, diagonal=-1).tolist() == [[0, 0, 0],
                                     [4, 0, 0],
                                     [7, 8, 0]]

# Compose with a constructor: ones cut to a triangle.
tri = t.triu(t.ones((3, 3)))
assert tri.tolist() == [[1.0, 1.0, 1.0],
                        [0.0, 1.0, 1.0],
                        [0.0, 0.0, 1.0]]
print("triu(z)")
print(t.triu(z))
print("tril(z, diagonal=-1) — strictly below")
print(t.tril(z, diagonal=-1))
print("triu(ones)")
print(tri)




Why: "strictly above/below" is expressed by shifting the cut (k=1 / k=-1),
not by post-processing. Most triangle tasks are one call with the right k.


<!-- dd:dd-q31 -->

### Problem 31 · faded — your turn

The n×n upper-triangular matrix of ones (diagonal included).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 1., 1., 1.],
        [0., 1., 1., 1.],
        [0., 0., 1., 1.],
        [0., 0., 0., 1.]])
```


In [ ]:
import torch as t

def solve(n):
    """Upper triangle of ones, including the main diagonal."""
    return t._____(t._____((n, n)))


# Example run — the grader calls solve() with several different values of n,
# including edge cases. Your function must work for all of them.
example = 4
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(31)


In [ ]:
#@title 💡 Solution — Problem 31
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.triu(t.ones((n, n)))


example = 4
print(solve(example))


<!-- dd:dd-q78 -->

### Problem 78 · guided

Write a function solve(n) that takes an integer n >= 1 and returns an n x n BOOLEAN PyTorch tensor in which the upper-triangular part including the main diagonal is True and everything strictly below the diagonal is False. The dtype must be bool, not integers.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ True,  True,  True],
        [False,  True,  True],
        [False, False,  True]])
```


<details>
<summary>Hints</summary>

1. Upper-triangular INCLUDING the diagonal is the default offset — no
   `diagonal=` argument needed.
2. Build a full tensor of the right dtype first, then let the triangular
   helper zero out the lower part.
3. `t.triu(t.ones((n, n), dtype=t.bool))` — set the dtype at construction;
   converting afterwards is an extra pass, and an int tensor would fail
   the bool check.

</details>


In [ ]:
import torch as t

def solve(n):
    """Return an n x n bool array: True on and above the diagonal."""
    return None


# Example run — the grader calls solve() with several sizes.
print(solve(3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(78)


In [ ]:
#@title 💡 Solution — Problem 78
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.triu(t.ones((n, n), dtype=t.bool))


print(solve(3))


<!-- dd:dd-q140 -->

### Problem 140 · independent

Write a function solve(z, k) that takes a 2-D square integer tensor and an integer offset k (positive, zero, or negative), and returns the SUM of the entries on the k-th diagonal — the main diagonal for k = 0, above it for positive k, below it for negative k — as a plain Python int. (An earlier drill extracts the diagonal; this one reduces it.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
12
```


In [ ]:
import torch as t

def solve(z, k):
    """Return the sum of z's k-th diagonal as a plain int."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.arange(1, 10).reshape(3, 3)
print(solve(example, -1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(140)


In [ ]:
#@title 💡 Solution — Problem 140
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, k):
    return int(t.diag(z, diagonal=k).sum())


example = t.arange(1, 10).reshape(3, 3)
print(solve(example, -1))


<!-- dd:dd-q16 -->

### Problem 16 · independent

Write a function solve(n) that takes an integer n >= 1 and returns an n x n floating-point matrix whose entries strictly below the main diagonal are 1.0 and all other entries (the diagonal itself and above) are 0.0. For n = 1 the result is [[0.0]]. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.]])
```


In [ ]:
import torch as t

def solve(n):
    """Return an n x n float matrix: 1.0 strictly below the diagonal, else 0.0."""
    return None


# Example run — the grader calls solve() with several sizes.
print(solve(5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(16)


In [ ]:
#@title 💡 Solution — Problem 16
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.tril(t.ones((n, n)), diagonal=-1)


print(solve(5))


<!-- dd:dd-q4 -->

### Problem 4 · independent

Write a function solve(n) that takes an integer n >= 1 and returns an n x n integer matrix that is all zeros except immediately below the main diagonal, where the values 1, 2, ..., n-1 appear in order. For n = 1 there is no room below the diagonal, so the result is just [[0]].

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0],
        [0, 2, 0, 0, 0],
        [0, 0, 3, 0, 0],
        [0, 0, 0, 4, 0]])
```


In [ ]:
import torch as t

def solve(n):
    """Return an n x n matrix with 1..n-1 just below the main diagonal."""
    return None


# Example run — the grader calls solve() with several sizes.
print(solve(5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(4)


In [ ]:
#@title 💡 Solution — Problem 4
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.diag(1 + t.arange(n - 1), diagonal=-1)


print(solve(5))


<!-- dd:dd-q160 -->

### Problem 160 · independent

Write a function solve(v) that takes a 1-D PyTorch tensor v of length n and returns an n x n lower-triangular matrix whose row i contains the first i+1 entries of v followed by zeros: L[i, j] = v[j] when j <= i and 0 otherwise. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 0, 0],
        [1, 2, 0],
        [1, 2, 3]])
```


In [ ]:
import torch as t

def solve(v):
    """Return the lower-triangular matrix with v's prefixes as rows."""
    return None


# Example run — the grader calls solve() with several vectors.
print(solve(t.tensor([1, 2, 3])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(160)


In [ ]:
#@title 💡 Solution — Problem 160
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v):
    n = v.numel()
    return t.tril(t.tile(v, (n, 1)))


print(solve(t.tensor([1, 2, 3])))


<!-- dd:dd-q180 -->

### Problem 180 · independent

Write a function solve(n, d0, d1) that takes a size n and two scalars, and returns an n x n float matrix with d0 at every position of the main diagonal and d1 on both diagonals directly above and below it, all other entries zero — the classic tridiagonal stencil matrix.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 5., -1.,  0.,  0.],
        [-1.,  5., -1.,  0.],
        [ 0., -1.,  5., -1.],
        [ 0.,  0., -1.,  5.]])
```


In [ ]:
import torch as t

def solve(n, d0, d1):
    """Return the n x n tridiagonal matrix with d0 on and d1 beside the diagonal."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(4, 5.0, -1.0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(180)


In [ ]:
#@title 💡 Solution — Problem 180
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(n, d0, d1):
    m = t.zeros(n, n)
    i = t.arange(n)
    m[i, i] = d0
    j = t.arange(n - 1)
    m[j, j + 1] = d1
    m[j + 1, j] = d1
    return m


print(solve(4, 5.0, -1.0))


#### Common mistakes

- **"t.diag always extracts."** — With 1-D input it BUILDS a matrix. The
  function is two-way, switched by input rank; a stray reshape can flip you
  into the wrong mode.
- **"triu deletes the diagonal too."** — Default k=0 KEEPS the diagonal.
  "Strictly above/below" needs k=1 / k=-1 — read the task for whether the
  diagonal is in or out.
- **"Positive k means below."** — Positive is ABOVE the main diagonal,
  negative below, consistently across diag/trace/triu/tril/eye. When unsure,
  test on `t.arange(9).reshape(3,3)` where every entry announces its
  position.


<!-- dd:dd-kp-numpy-argsort-ranking -->

## Order statistics — argsort, ranks, sort-by

`numpy.argsort-ranking`


### sort-by — argsort as a row order


`t.sort` rearranges values. **`t.argsort` returns the indices that WOULD
sort them** — and that indirection is the tool.

First use: **sort one thing by another ("sort-by")**. To reorder a matrix's
ROWS so that column k comes out ascending: `z[z[:, k].argsort()]`. Read it
inside out — argsort of the key column gives the row order; fancy indexing
applies that order to whole rows. The rows travel intact; only their sequence
changes. Any "sort records by field" task is this two-step.

Note what is NOT happening: no row contents are sorted — `t.sort(z, axis=0)`
would destroy the records by sorting each column independently.


In [ ]:
import torch as t

z = t.tensor([[10, 3],
              [20, 1],
              [30, 2]])

# Sort ROWS by column 1. Inside out:
key = z[:, 1]                    # the key column: [3, 1, 2]
order = key.argsort()            # row order that sorts it: [1, 2, 0]
sorted_rows = z[order]           # fancy indexing moves whole rows
assert sorted_rows.tolist() == [[20, 1], [30, 2], [10, 3]]
# One-liner form you'll actually write: z[z[:, 1].argsort()]
print("key column", key, "-> row order", order)
print(sorted_rows)




Why: unpacking the sort-by into key/order/apply once makes the one-liner
readable forever after.


<!-- dd:dd-q92 -->

### Problem 92 · faded — your turn

Reorder rows so that column k is ascending.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 9],
        [5, 2]])
```


In [ ]:
import torch as t

def solve(z, k):
    """Rows of z reordered so column k comes out ascending."""
    return z[z[:, k]._____()]


# Example run — the grader calls solve() with several inputs.
example = t.tensor([[5, 2], [1, 9]])
print(solve(example, 0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(92)


In [ ]:
#@title 💡 Solution — Problem 92
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, k):
    return z[z[:, k].argsort()]


example = t.tensor([[5, 2], [1, 9]])
print(solve(example, 0))


### ranks — argsort twice


**Ranks.** The composition `t.argsort(t.argsort(z))` assigns each element
its 0-based rank (0 = smallest). First argsort: "who is in each sorted
position"; second: "what position does each element hold" — the inverse
permutation. For distinct values this is THE rank formula worth memorizing.

One argsort is *not* ranks: it answers "which element is at sorted position
i?", ranks answer the inverse question — hence argsort twice.


In [ ]:
import torch as t

# Ranks (0 = smallest) via double argsort — the inverse permutation.
v = t.tensor([30.0, 10.0, 20.0])
ranks = t.argsort(t.argsort(v))
assert ranks.tolist() == [2, 0, 1]       # 30 is largest -> rank 2
print("values", v)
print("ranks ", ranks, " (0 = smallest)")




Why: trace one element — 30.0 sits at sorted position 2, so its rank is 2.
The first argsort maps positions→elements; the second inverts it to
elements→positions.


<!-- dd:dd-q114 -->

### Problem 114 · faded — your turn

Each element's 0-based rank (values distinct), reported at its own position.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0, 3, 1, 2])
```


In [ ]:
import torch as t

def solve(z):
    """0-based rank of each element: smallest -> 0, at each element's slot."""
    return z._____()._____()


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([10, 40, 20, 30])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(114)


In [ ]:
#@title 💡 Solution — Problem 114
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.argsort().argsort()


print(solve(t.tensor([10, 40, 20, 30])))


### top-k — the tail of an argsort


**Top-k.** The k largest values' indices: `t.argsort(z)[-k:][::-1]`
(ascending order → take the tail → flip to largest-first). When k is small
and n is huge, `t.topk(z, k)` finds the top-k in linear
time — partial ordering for free; sort just those k afterwards if order
matters.

Direction control, since argsort has no `descending=`: argsort the negated
array (`t.argsort(-z)`) or flip the result (`[::-1]`) — same options as
sort.


In [ ]:
import torch as t

# Indices of the top-2 values, largest first.
w = t.tensor([5.0, 9.0, 1.0, 7.0])
top2 = t.argsort(w)[-2:].flip(0)
assert top2.tolist() == [1, 3]           # 9.0 at index 1, then 7.0 at 3
assert w[top2].tolist() == [9.0, 7.0]    # indices recover the values
print("w", w)
print("top-2 indices", top2, "-> values", w[top2])




Why: each stage of the chain is checkable — ascending indices, keep the last
k, reverse. When the task says "any order is fine", argpartition saves the
final sort.


<!-- dd:dd-q119 -->

### Problem 119 · faded — your turn

Indices of the k largest elements, largest first (values distinct).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 3])
```


In [ ]:
import torch as t

def solve(z, k):
    """Indices of the k largest entries of z, largest first."""
    return t._____(z)[-k:]_____


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([5.0, 9.0, 1.0, 7.0]), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(119)


In [ ]:
#@title 💡 Solution — Problem 119
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z, k):
    return t.topk(z, k).indices


print(solve(t.tensor([5.0, 9.0, 1.0, 7.0]), 2))


<!-- dd:dd-q525 -->

### Problem 525 · guided

Write a function solve(z, k) that takes a 2-D tensor z and a column index k, and returns z's ROWS reordered so that column k comes out DESCENDING (largest first). Each row must keep its own contents — only the order of the rows changes. Build the ascending row order with argsort on the key column and reverse it before indexing. Sorting the columns independently would tear the rows apart.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[10,  3],
        [30,  2],
        [20,  1]])
```


<details>
<summary>Hints</summary>

1. The sort-by recipe is unchanged — argsort the key column, fancy-index the
   rows. Only the DIRECTION differs, and direction is a property of the
   order, not of the indexing.
2. `z[:, k].argsort()` is the ascending row order; reversing an order is the
   same `.flip(0)` the top-k segment used.
3. `z[z[:, k].argsort().flip(0)]` — one expression, rows still intact.

</details>


In [ ]:
import torch
import torch as t

def solve(z, k):
    """Rows of z reordered so column k is descending."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[10, 3], [20, 1], [30, 2]]), 1)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(525)


In [ ]:
#@title 💡 Solution — Problem 525
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(z, k):
    """Rows of z reordered so column k is descending."""
    order = z[:, k].argsort().flip(0)
    return z[order]


example = (t.tensor([[10, 3], [20, 1], [30, 2]]), 1)
print(solve(*example))


<!-- dd:dd-q183 -->

### Problem 183 · independent

Write a function solve(z) that takes a 2-D float tensor with DISTINCT values within each row and returns an integer tensor of the same shape where each entry is replaced by its 0-based rank WITHIN ITS ROW (0 = the row's smallest). Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2, 0, 1]])
```


In [ ]:
import torch as t

def solve(z):
    """Return each entry's 0-based rank within its own row."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[0.3, 0.1, 0.2]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(183)


In [ ]:
#@title 💡 Solution — Problem 183
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.argsort(dim=1).argsort(dim=1)


print(solve(t.tensor([[0.3, 0.1, 0.2]])))


#### Common mistakes

- **"argsort returns sorted values."** — It returns INDICES. `z[t.argsort(z)]`
  is the sorted array; the indices themselves are the tool for sort-by, ranks,
  and top-k.
- **"Sorting a table by a column = t.sort(z, axis=0)."** — That sorts every
  column independently, tearing rows apart. Row-preserving sort-by is
  argsort-the-key + fancy-index-the-rows.
- **"One argsort gives ranks."** — One argsort answers "which element is at
  sorted position i?" Ranks answer the inverse question, hence argsort twice.
  For distinct values they coincide only when the array was already sorted.


<!-- dd:dd-kp-numpy-nan-handling -->

## NaN and Inf — detecting and repairing

`numpy.nan-handling`


**NaN ("not a number")** is the float value that marks missing or undefined
results (0/0, missing sensor readings). It has one property that breaks naive
code: **NaN compares unequal to everything, including itself.** `x == t.nan`
is ALWAYS False — so you cannot find NaNs with `==`.

The dedicated predicates are the way in — each returns a boolean mask:

- **`t.isnan(x)`** — True where the entry is NaN.
- **`t.isinf(x)`** — True where it is +Inf or −Inf (which are *not* NaN:
  they come from 1/0-style overflow and DO behave in comparisons).
- **`t.isfinite(x)`** — True where the entry is an ordinary number
  (not NaN, not ±Inf). Often the cleanest: "keep the good rows" is a
  condition on isfinite.

From the mask, the standard moves are the ones you already own:

- **Detect**: `t.isnan(x).any()` — is anything missing?
- **Repair**: masked assignment `x[t.isnan(x)] = fill` on a copy, or the
  packaged `t.nan_to_num(z, nan=0.0)` (which returns a new array and can
  also replace ±Inf via `posinf=`/`neginf=`).
- **Skip**: the `nan*` reduction family — `t.nansum`, `t.nanmean`,
  `t.nanmedian` — compute as if NaNs weren't there (a plain `mean` of data
  containing NaN is NaN, since NaN propagates through arithmetic).

On 2-D data, combining an isnan mask with `any(axis=...)` answers per-row /
per-column questions ("which columns contain a NaN?") — the axis mechanics
get full treatment in the broadcasting lesson, but the pattern is worth
seeing here.


Task: check a vector for missing values; repair them to 0.0 without touching
the input; take a NaN-ignoring mean.


In [ ]:
import torch as t

x = t.tensor([0.5, t.nan, 1.0])

# Detection MUST go through isnan — == can't see NaN, even against itself.
assert not (x == t.nan).any()           # always False everywhere: useless
assert t.isnan(x).any()                 # the real test
assert bool(t.isnan(x).any()) is True   # plain-bool contract at the boundary

# Repair, packaged: new array, NaNs -> 0.0, everything else unchanged.
fixed = t.nan_to_num(x, nan=0.0)
assert fixed.tolist() == [0.5, 0.0, 1.0]
assert t.isnan(x[1])                    # input untouched

# Repair, by hand — same result, and the form that generalizes to
# arbitrary fills (e.g. the mean of the good entries):
out = x.clone()
out[t.isnan(out)] = 0.0
assert out.tolist() == [0.5, 0.0, 1.0]

# NaN poisons plain reductions; the nan* family ignores it.
assert t.isnan(x.mean())                # NaN propagates
assert t.nanmean(x) == 0.75             # mean of 0.5 and 1.0
print("x            ", x)
print("x == t.nan   ", x == t.nan, " <- never True, not even at the NaN")
print("t.isnan(x)   ", t.isnan(x))
print("nan_to_num   ", fixed)
print("mean", x.mean().item(), "| nanmean", t.nanmean(x).item())




Why each step:

1. Running the broken `==` check next to `isnan` once is the fastest
   inoculation against the classic bug — the comparison silently returns all
   False rather than erroring.
2. `nan_to_num` vs manual masked assignment: the packaged call for standard
   fills, the manual pattern when the fill is computed (column means, medians
   — see the drills).
3. `x.mean()` coming out NaN is not an error, it's NaN doing its job of
   propagating; deciding between "propagate" and "ignore" (`nanmean`) is a
   data-semantics choice the task will state.


<!-- dd:dd-q32 -->

### Problem 32 · faded — your turn

Does the array contain any NaN? (plain Python bool)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
True
```


In [ ]:
import torch as t

def solve(x):
    """True iff any entry of x is NaN."""
    return bool(t._____(x).any())


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.0, t.nan, 3.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(32)


In [ ]:
#@title 💡 Solution — Problem 32
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return bool(t.isnan(x).any())


example = t.tensor([1.0, t.nan, 3.0])
print(solve(example))


<!-- dd:dd-q18 -->

### Problem 18 · guided

Write a function solve(z) that takes a 1-D PyTorch tensor of floats that may contain NaN entries (but no infinities) and returns a new tensor in which every NaN has been replaced by 0.0 while every other value is unchanged. Do not modify the input tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.5000, 0.0000, 1.0000])
```


<details>
<summary>Hints</summary>

1. Every NaN becomes 0.0, everything else unchanged, input not modified —
   detection then repair.
2. There is a single packaged function whose keyword is literally `nan=`.
3. `t.nan_to_num(z, nan=0.0)` — or the copy + masked-assignment form; both
   satisfy "do not modify the input" (why?).

</details>


In [ ]:
import torch as t

def solve(z):
    """Return z with every NaN replaced by 0.0."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([0.5, t.nan, 1.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(18)


In [ ]:
#@title 💡 Solution — Problem 18
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.nan_to_num(z, nan=0.0)


example = t.tensor([0.5, t.nan, 1.0])
print(solve(example))


<!-- dd:dd-q115 -->

### Problem 115 · independent

Write a function solve(z) that takes a 2-D PyTorch float tensor that may contain NaN entries and returns a 1-D boolean tensor with one entry per COLUMN: True where that column consists entirely of NaN values, False otherwise. The result's dtype must be bool.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([False,  True])
```


In [ ]:
import torch as t

def solve(z):
    """Return per-column flags: True where the whole column is NaN."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([[0.0, t.nan], [1.0, t.nan]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(115)


In [ ]:
#@title 💡 Solution — Problem 115
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.isnan(z).all(dim=0)


example = t.tensor([[0.0, t.nan], [1.0, t.nan]])
print(solve(example))


<!-- dd:dd-q142 -->

### Problem 142 · independent

Write a function solve(z) that takes a 2-D PyTorch tensor of floats which may contain NaN, Inf, or -Inf entries. Return a 1-D tensor of the row indices, in increasing order, of every row that contains at least one non-finite value. If every entry is finite, return an empty 1-D tensor. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 3])
```


In [ ]:
import torch as t

def solve(z):
    """Return the 1-D array of indices of rows in z that contain any NaN or Inf."""
    return None


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 2.0, 3.0],
                    [4.0, t.nan, 6.0],
                    [7.0, 8.0, 9.0],
                    [t.inf, 10.0, 11.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(142)


In [ ]:
#@title 💡 Solution — Problem 142
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.where(~t.isfinite(z).all(dim=1))[0]


example = t.tensor([[1.0, 2.0, 3.0],
                    [4.0, t.nan, 6.0],
                    [7.0, 8.0, 9.0],
                    [t.inf, 10.0, 11.0]])
print(solve(example))


<!-- dd:dd-q120 -->

### Problem 120 · independent

Write a function solve(x) that takes a 2-D PyTorch float tensor in which some entries are NaN (but no column is entirely NaN) and returns a new tensor where every NaN has been replaced by the mean of the NON-NaN values in its own column. Finite entries are unchanged. Do not modify the input.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 4.],
        [3., 4.]])
```


In [ ]:
import torch as t

def solve(x):
    """Return x with each NaN replaced by its column's non-NaN mean."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([[1.0, t.nan], [3.0, 4.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(120)


In [ ]:
#@title 💡 Solution — Problem 120
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(x):
    out = x.clone()
    col_means = t.nanmean(out, dim=0)
    idx = t.nonzero(t.isnan(out), as_tuple=True)
    out[idx] = col_means[idx[1]]
    return out


example = t.tensor([[1.0, t.nan], [3.0, 4.0]])
print(solve(example))


<!-- dd:dd-q170 -->

### Problem 170 · independent

Write a function solve(z) that takes a 2-D float tensor containing NaN entries (no column entirely NaN) and returns a new tensor with every NaN replaced by the mean of the NON-NaN values in its own COLUMN, computed with keepdims so the statistics broadcast cleanly. Finite entries stay unchanged; the input must not be modified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 6.],
        [3., 6.]])
```


In [ ]:
import torch as t

def solve(z):
    """Return z with NaNs replaced by their column's non-NaN mean."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1.0, t.nan], [3.0, 6.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(170)


In [ ]:
#@title 💡 Solution — Problem 170
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    means = t.nanmean(z, dim=0, keepdim=True)
    return t.where(t.isnan(z), means, z)


print(solve(t.tensor([[1.0, t.nan], [3.0, 6.0]])))


#### Common mistakes

- **"Find NaNs with `x == t.nan`."** — NaN ≠ NaN by IEEE definition, so that
  comparison is False everywhere. Only `t.isnan` detects them.
- **"Inf is a kind of NaN."** — Different animals: Inf is a well-ordered
  value (`t.inf > 1e300` is True) from overflow/division-by-zero; NaN is
  unordered missingness. `isnan`, `isinf`, `isfinite` slice the three cases
  cleanly.
- **"mean() skips missing values like pandas."** — PyTorch propagates: one NaN
  makes the whole mean NaN. Skipping is opt-in via `t.nanmean` and friends.


<!-- dd:dd-kp-numpy-pad-borders -->

## Borders and padding

`numpy.pad-borders`


Border tasks come in two mirror-image forms, each with its own tool:

**Growing — add a border AROUND the data: `t.nn.functional.pad`.**

> `t.nn.functional.pad(z, (1, 1, 1, 1), mode="constant", value=0)`

wraps `z` in a one-cell-thick frame of zeros: shape (r, c) → (r+2, c+2).

Mind the argument order — it is the one real trap on this page. The pad
tuple runs **last dimension first**, in (before, after) pairs:
`(left, right, top, bottom)` for a 2-D tensor. Pass a shorter tuple and only
the trailing dimensions get padded, which is a quiet way to pad columns and
wonder where your rows went. `value` sets the fill, and other modes
(`replicate`, `reflect`, `circular`) extend the data instead of a constant —
padding is the standard prelude to sliding-window and stencil operations,
where the window must not fall off the edge.

**Marking — modify the border WITHIN the existing shape: slice assignment.**
The frame of a matrix is four slices, or — the cleaner inverse — everything
*except* the frame is ONE slice, the interior `z[1:-1, 1:-1]`. So:

- ring of ones, hollow inside: start from `t.ones`, zero the interior:
  `z[1:-1, 1:-1] = 0`.
- overwrite the frame with a value: start from the data, assign the four
  edge slices `z[0, :]`, `z[-1, :]`, `z[:, 0]`, `z[:, -1]`.

The interior-slice trick has a built-in kindness: for 1×n or 2×n arrays,
`z[1:-1, 1:-1]` is an *empty* selection, so the assignment quietly does
nothing — exactly the right behavior when "everything is border", no special
case needed.

Choosing: does the output's shape GROW (pad) or stay the same (slice
assignment)? Read the task's shape contract first.


Task: (a) surround a matrix of ones with a zero border — shape grows;
(b) build a same-shape "picture frame": ones on the border, zeros inside.


In [ ]:
import torch as t

z = t.ones((2, 2))

# (a) GROW: pad adds cells around the data. (2,2) -> (4,4).
framed = t.nn.functional.pad(z, (1, 1, 1, 1), mode="constant", value=0)
assert tuple(framed.shape) == (4, 4)
assert framed.tolist() == [[0.0, 0.0, 0.0, 0.0],
                           [0.0, 1.0, 1.0, 0.0],
                           [0.0, 1.0, 1.0, 0.0],
                           [0.0, 0.0, 0.0, 0.0]]

# (b) SAME SHAPE: build all-ones, then blank the interior with ONE slice.
n = 4
ring = t.ones((n, n))
ring[1:-1, 1:-1] = 0.0            # interior = rows 1..-2, cols 1..-2
assert ring.tolist() == [[1.0, 1.0, 1.0, 1.0],
                         [1.0, 0.0, 0.0, 1.0],
                         [1.0, 0.0, 0.0, 1.0],
                         [1.0, 1.0, 1.0, 1.0]]

# The degenerate case is free: a 2x2 has no interior, so nothing changes.
small = t.ones((2, 2))
small[1:-1, 1:-1] = 0.0           # empty selection — assignment is a no-op
assert small.tolist() == [[1.0, 1.0], [1.0, 1.0]]
print("(a) padded — shape grew to", tuple(framed.shape))
print(framed)
print("(b) same shape, interior blanked")
print(ring)
print("(c) 2x2 has no interior, so nothing changed:", small.tolist())




Why each step:

1. In (a) note who supplies the border values: `pad` does — the original
   tensor is unchanged in the middle. Padding never mutates; it returns a
   new, larger tensor. The four 1s are (left, right, top, bottom).
2. In (b) the insight is *invert the selection*: four border slices are
   fiddly, one interior slice is clean. `1:-1` reads "skip the first, skip
   the last" on each axis.
3. The 2×2 no-op is why the interior-slice formulation is preferred over
   explicit border slices in tasks that say "for n = 1 or 2 the whole matrix
   is border": empty slices make the edge cases disappear.


<!-- dd:dd-q17 -->

### Problem 17 · faded — your turn

Zero border AROUND the data: (r, c) → (r+2, c+2).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 0., 0., 0.],
        [0., 1., 1., 0.],
        [0., 1., 1., 0.],
        [0., 0., 0., 0.]])
```


In [ ]:
import torch as t

def solve(z):
    """Surround z with a one-cell-thick border of zeros."""
    return t._____.functional.pad(z, _____, _____="constant", _____=_____)


# Example run — the grader calls solve() with several arrays.
example = t.ones((2, 2))
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(17)


In [ ]:
#@title 💡 Solution — Problem 17
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z):
    return t.nn.functional.pad(z, (1, 1, 1, 1), mode="constant", value=0)


example = t.ones((2, 2))
print(solve(example))


<!-- dd:dd-q27 -->

### Problem 27 · guided

Write a function solve(n) that takes a positive integer n and returns an n x n PyTorch tensor of floats whose outer ring — the first and last row and the first and last column — is 1.0, and whose interior block is 0.0. Use slice assignment rather than Python loops. Note that for n = 1 or n = 2 every entry lies on the border, so the whole matrix should be 1.0.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 1., 1., 1.],
        [1., 0., 0., 1.],
        [1., 0., 0., 1.],
        [1., 1., 1., 1.]])
```


<details>
<summary>Hints</summary>

1. Ones on the outer ring, zeros inside, same shape — grow or modify?
2. Build the ones first; the interior is a single two-axis slice.
3. `z[1:-1, 1:-1] = 0.0` — and check the task's promise about n=1, n=2
   against what an empty slice assignment does.

</details>


In [ ]:
import torch as t

def solve(n):
    """Return an n x n float array with a border of 1.0 and an interior of 0.0."""
    return None


# Example run — the grader calls solve() with several different values of n,
# including edge cases. Your function must work for all of them.
print(solve(4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(27)


In [ ]:
#@title 💡 Solution — Problem 27
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    z = t.ones((n, n))
    z[1:-1, 1:-1] = 0.0
    return z


print(solve(4))


<!-- dd:dd-q90 -->

### Problem 90 · independent

Write a function solve(z, fill) that takes a 2-D PyTorch tensor z and a scalar fill, and returns a new tensor with a one-element-wide border of fill surrounding z on all four sides. Shape (r, c) becomes (r+2, c+2), with the original values in the interior.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[-1, -1, -1, -1],
        [-1,  1,  1, -1],
        [-1,  1,  1, -1],
        [-1, -1, -1, -1]])
```


In [ ]:
import torch as t

def solve(z, fill):
    """Return z surrounded by a one-cell border of the value fill."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.ones((2, 2), dtype=t.int64)
print(solve(example, -1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(90)


In [ ]:
#@title 💡 Solution — Problem 90
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z, fill):
    return t.nn.functional.pad(z, (1, 1, 1, 1), mode="constant", value=fill)


example = t.ones((2, 2), dtype=t.int64)
print(solve(example, -1))


<!-- dd:dd-q186 -->

### Problem 186 · independent

Write a function solve(z, shape, position, fill) that takes a 2-D integer tensor z, an ODD-sized window shape (h, w), a center position (r, c) inside z, and a fill value. Return the h x w subarray of z CENTERED at that position; wherever the window extends past z's boundary, use the fill value.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[-1, -1, -1],
        [-1,  0,  1],
        [-1,  3,  4]])
```


In [ ]:
import torch as t

def solve(z, shape, position, fill):
    """Return the shape-sized window of z centered at position, padded with fill."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(9).reshape(3, 3), (3, 3), (0, 0), -1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(186)


In [ ]:
#@title 💡 Solution — Problem 186
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, shape, position, fill):
    out = t.full(shape, fill, dtype=z.dtype)
    for axis in range(2):
        pass
    h, w = shape
    r, c = position
    r_lo, c_lo = r - h // 2, c - w // 2
    src_r = slice(max(r_lo, 0), min(r_lo + h, z.shape[0]))
    src_c = slice(max(c_lo, 0), min(c_lo + w, z.shape[1]))
    dst_r = slice(src_r.start - r_lo, src_r.start - r_lo + (src_r.stop - src_r.start))
    dst_c = slice(src_c.start - c_lo, src_c.start - c_lo + (src_c.stop - src_c.start))
    out[dst_r, dst_c] = z[src_r, src_c]
    return out


print(solve(t.arange(9).reshape(3, 3), (3, 3), (0, 0), -1))


#### Common mistakes

- **"pad modifies in place."** — It returns a new, bigger tensor (it must —
  the shape changes). The original is one of its ingredients, not its victim.
- **"The pad tuple is (top, bottom, left, right)."** — It is LAST DIMENSION
  FIRST: `(left, right, top, bottom)` for 2-D. Getting this backwards pads a
  valid-looking tensor with the frame on the wrong axis.
- **"The border needs four assignments."** — Assigning the border is four
  slices, but most frame tasks invert: fill everything, then overwrite the
  single interior slice `z[1:-1, 1:-1]`.
- **"Small matrices need an if-statement."** — `z[1:-1, 1:-1]` on a 1×n or
  2×n array selects nothing, and assigning to an empty selection is a legal
  no-op. The slice formulation handles the degenerate cases by construction.


<!-- dd:dd-kp-numpy-index-grids -->

## Index-pattern grids — checkerboards and coordinate masks

`numpy.index-grids`


### periodic patterns — strided slice assignment


A family of tasks asks you to build or mask a matrix based on **each cell's
coordinates**: checkerboards, distance-from-center maps, bands around the
diagonal. When the pattern has a **fixed period**, strided slices do it.

Slices accept a step, so `z[::2]` is "every even row" and `z[1::2, ::2]` is
"odd rows, even columns". A checkerboard is exactly two such assignments on
a zeros canvas:

```python no-run
z[1::2, ::2] = 1    # odd rows, even columns
z[::2, 1::2] = 1    # even rows, odd columns
```

(`z[::2]` is every SECOND row, not the first two — `z[:2]` is that. Step
lives in the third slot: start:stop:step.)


In [ ]:
import torch as t

# Checkerboard, period-2 pattern -> strided slice assignment.
rows, cols = 3, 4
z = t.zeros((rows, cols), dtype=t.int64)
z[1::2, ::2] = 1        # cells where row is odd  and col is even
z[::2, 1::2] = 1        # cells where row is even and col is odd
assert z.tolist() == [[0, 1, 0, 1],
                      [1, 0, 1, 0],
                      [0, 1, 0, 1]]
# Sanity: (i + j) odd <=> cell is 1 — exactly "no equal neighbors".
print(z)




Why: the two assignments partition the 1-cells by row parity — walking one
cell ("row 1 is odd, col 0 is even → 1") verifies the slice choice faster
than staring at the pattern.


<!-- dd:dd-q2 -->

### Problem 2 · faded — your turn

Checkerboard of 0s and 1s, 0 in the top-left corner.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 0, 1],
        [1, 0, 1, 0],
        [0, 1, 0, 1]])
```


In [ ]:
import torch as t

def solve(rows, cols):
    """Checkerboard with z[0,0] == 0, alternating both directions."""
    z = t.zeros((rows, cols), dtype=t.int64)
    z[1::2, _____] = 1
    z[_____, 1::2] = 1
    return z


# Example run — the grader calls solve() with several different sizes.
print(solve(3, 4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(2)


In [ ]:
#@title 💡 Solution — Problem 2
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols):
    z = t.zeros((rows, cols), dtype=t.int64)
    z[1::2, ::2] = 1
    z[::2, 1::2] = 1
    return z


print(solve(3, 4))


### coordinate formulas — index-vector arithmetic


When the pattern is a **FORMULA in i and j**, build the row-index and
column-index vectors and lean on their shapes: `t.arange(n)[:, None]` is a
**column** `y` of shape (n, 1) and `t.arange(n)[None, :]` is a **row** `x` of
shape (1, n). Any arithmetic between them produces the full (n, n) matrix of
`f(i, j)` values — each cell computed from its own coordinates. (WHY a
(n,1)-by-(1,n) operation yields (n,n) is broadcasting, next lesson's opening
KP — here, use it as "the coordinate-grid recipe".) Examples:

- Manhattan distance from the center: `t.abs(y - c) + t.abs(x - c)`.
- Diagonal band mask: `t.abs(y - x) <= 1` — True within one step of the
  main diagonal; multiply by `z` or use as a mask to keep the band.
- "Every cell = its row index": `y + 0 * x`.

If you would rather have both coordinates as full (n, n) matrices,
`t.meshgrid(t.arange(n), t.arange(n), indexing="ij")` returns exactly that —
`indexing="ij"` is the row-major convention these tasks assume, and leaving
it off gives you the transpose.

The decision rule: periodic pattern → strided slices; coordinate formula →
index-vector arithmetic. Both build the structure without visiting cells in
Python.


In [ ]:
import torch as t

# Coordinate formula -> index vectors. y is a column (n,1), x a row (1,n).
n = 3
y = t.arange(n)[:, None]
x = t.arange(n)[None, :]
assert tuple(y.shape) == (3, 1) and tuple(x.shape) == (1, 3)
c = n // 2
manhattan = t.abs(y - c) + t.abs(x - c)
assert manhattan.tolist() == [[2, 1, 2],
                              [1, 0, 1],
                              [2, 1, 2]]

# Band mask: cell (i, j) survives iff |i - j| <= 1.
m = t.arange(16).reshape(4, 4)
band = t.abs(t.arange(4)[:, None] - t.arange(4)[None, :]) <= 1
kept = m * band
assert kept.tolist() == [[0, 1, 0, 0],
                         [4, 5, 6, 0],
                         [0, 9, 10, 11],
                         [0, 0, 14, 15]]
print("y shape", tuple(y.shape), "x shape", tuple(x.shape),
      "-> broadcast to", tuple(manhattan.shape))
print("manhattan distance from centre")
print(manhattan)
print("band mask |i - j| <= 1, applied")
print(kept)




Why: the SHAPES carry the meaning — the column `y` varies down, the row `x`
varies across, and combining them touches every (i, j) pair exactly once.
The band example shows formula→mask→apply; diagonal bands, wedges, and
"within k of the anti-diagonal" all fall to the same three moves.


<!-- dd:dd-q116 -->

### Problem 116 · faded — your turn

Manhattan distance of each cell from the center of an odd n×n grid.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2, 1, 2],
        [1, 0, 1],
        [2, 1, 2]])
```


In [ ]:
import torch as t

def solve(n):
    """Entry [i, j] = |i - n//2| + |j - n//2|."""
    y = t.arange(n)[:, None]
    x = t.arange(n)[None, :]
    c = n // 2
    return t.abs(y - c) _____ t.abs(x - c)


# Example run — the grader calls solve() with several sizes.
print(solve(3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(116)


In [ ]:
#@title 💡 Solution — Problem 116
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(n):
    y = t.arange(n)[:, None]
    x = t.arange(n)[None, :]
    c = n // 2
    return (y - c).abs() + (x - c).abs()


print(solve(3))


<!-- dd:dd-q73 -->

### Problem 73 · guided

Write a function solve(z) that takes a 2-D PyTorch tensor and returns a list of ((row, col), value) pairs covering every element in row-major order — reading across the first row, then the second, and so on. Each value must be a plain Python number, not a 0-dimensional tensor. PyTorch has no ndenumerate, so build the index pairs yourself from the tensor's shape.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[((0, 0), 0), ((0, 1), 1), ((1, 0), 2), ((1, 1), 3)]
```


<details>
<summary>Hints</summary>

1. Torch has no `ndenumerate`, so the (row, col) pairs have to come from
   the shape itself.
2. Row-major order is just the outer loop over rows and the inner loop
   over columns — a nested comprehension gives that ordering for free.
3. `rows, cols = z.shape`, then `[((i, j), z[i, j].item()) for i in
   range(rows) for j in range(cols)]`. `.item()` is required: the drill
   wants plain Python numbers, not 0-d tensors.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return [((row, col), value), ...] for every element of z, row-major."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(4).reshape(2, 2)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(73)


In [ ]:
#@title 💡 Solution — Problem 73
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    rows, cols = z.shape
    return [((i, j), z[i, j].item()) for i in range(rows) for j in range(cols)]


example = t.arange(4).reshape(2, 2)
print(solve(example))


<!-- dd:dd-q112 -->

### Problem 112 · independent

Write a function solve(z) that takes a square 2-D PyTorch tensor and returns a new tensor of the same shape in which entries within one position of the main diagonal (the diagonal itself plus the two adjacent diagonals) keep their values and every other entry is zero. Do not modify the input.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 0,  1,  0,  0],
        [ 4,  5,  6,  0],
        [ 0,  9, 10, 11],
        [ 0,  0, 14, 15]])
```


In [ ]:
import torch as t

def solve(z):
    """Return z with everything more than 1 off the main diagonal zeroed."""
    return None


# Example run — the grader calls solve() with several matrices.
example = t.arange(16).reshape(4, 4)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(112)


In [ ]:
#@title 💡 Solution — Problem 112
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    n = z.shape[0]
    mask = t.abs(t.arange(n)[:, None] - t.arange(n)[None, :]) <= 1
    return z * mask


example = t.arange(16).reshape(4, 4)
print(solve(example))


<!-- dd:dd-q72 -->

### Problem 72 · independent

Write a function solve(rows, cols, top_left) that takes two positive integers and a corner value top_left (0 or 1), and returns a rows x cols integer checkerboard: the entry at [0, 0] equals top_left, and values alternate between 0 and 1 along every row and every column so no two adjacent entries match.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 0, 1],
        [0, 1, 0],
        [1, 0, 1]])
```


In [ ]:
import torch as t

def solve(rows, cols, top_left):
    """Return a rows x cols checkerboard whose [0,0] entry is top_left."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(3, 3, 1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(72)


In [ ]:
#@title 💡 Solution — Problem 72
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(rows, cols, top_left):
    grid = t.stack(t.meshgrid(t.arange(rows), t.arange(cols), indexing="ij"))
    return (grid.sum(dim=0) + top_left) % 2


print(solve(3, 3, 1))


<!-- dd:dd-q11 -->

### Problem 11 · independent

Write a function solve(rows, cols) that takes two positive integers and returns a rows x cols integer matrix in which every entry equals its own row index: the top row is all 0s, the next row all 1s, and so on. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 0, 0, 0],
        [1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]])
```


In [ ]:
import torch as t

def solve(rows, cols):
    """Return a rows x cols matrix where each entry equals its row index."""
    return None


# Example run — the grader calls solve() with several sizes.
print(solve(4, 4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(11)


In [ ]:
#@title 💡 Solution — Problem 11
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(rows, cols):
    return t.meshgrid(t.arange(rows), t.arange(cols), indexing="ij")[0]


print(solve(4, 4))


<!-- dd:dd-q164 -->

### Problem 164 · independent

Write a function solve(c) that takes a 1-D PyTorch tensor c of length n and returns the n x n symmetric Toeplitz matrix T with T[i, j] = c[|i - j|]: c[0] runs down the main diagonal, c[1] along both first off-diagonals, and so on. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[10, 20, 30],
        [20, 10, 20],
        [30, 20, 10]])
```


In [ ]:
import torch as t

def solve(c):
    """Return the symmetric Toeplitz matrix T[i,j] = c[|i-j|]."""
    return None


# Example run — the grader calls solve() with several vectors.
print(solve(t.tensor([10, 20, 30])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(164)


In [ ]:
#@title 💡 Solution — Problem 164
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c):
    n = c.numel()
    i = t.arange(n)[:, None]
    j = t.arange(n)[None, :]
    return c[t.abs(i - j)]


print(solve(t.tensor([10, 20, 30])))


#### Common mistakes

- **"Patterned matrices need nested loops."** — Periodic patterns are strided
  slice assignments; coordinate formulas are index-vector arithmetic.
  Python-level cell visits are never required in this family.
- **"The coordinate vectors have to be full matrices."** — They are SKINNY —
  shape (n, 1) and (1, n) — and expand only when combined, so the (n, n)
  grid never exists in memory. `t.meshgrid` builds the full pair when you
  actually want it.
- **"`z[::2]` means the first two rows."** — It means every SECOND row
  (step 2, from row 0). `z[:2]` is the first two. Step lives in the third
  slot: start:stop:step.
